In [3]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/input/competitions/constituency-war-room-grievance-triage-challenge"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(
    f"{DATA_DIR}/sample_submission.csv"
)

print("TRAIN:", train.shape)
print("TEST:", test.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nTest columns:")
print(test.columns.tolist())

print("\nLabels:", train["label"].nunique())

print("\nMissing values:")
print(train.isna().sum())

print("\nDuplicate complete rows:")
print(train.duplicated().sum())

print("\nDuplicate bodies:")
print(train["body"].duplicated().sum())

print("\nDuplicate subjects:")
print(train["subject"].duplicated().sum())

TRAIN: (4800, 7)
TEST: (2000, 6)

Train columns:
['id', 'subject', 'body', 'channel', 'district', 'complaint_history', 'label']

Test columns:
['id', 'subject', 'body', 'channel', 'district', 'complaint_history']

Labels: 24

Missing values:
id                   0
subject              0
body                 0
channel              0
district             0
complaint_history    0
label                0
dtype: int64

Duplicate complete rows:
0

Duplicate bodies:
0

Duplicate subjects:
3982


In [4]:
# ============================================================
# STEP 2 — SUBJECT TEMPLATE ANALYSIS
# ============================================================

from collections import Counter

# ------------------------------------------------------------
# 1. Basic subject statistics
# ------------------------------------------------------------

subject_counts = train["subject"].value_counts()

print("Total training rows:", len(train))
print("Unique subjects:", train["subject"].nunique())
print("Repeated subjects:", (subject_counts > 1).sum())
print("Subjects appearing once:", (subject_counts == 1).sum())

print("\nTop 20 most frequent subjects:")
display(subject_counts.head(20).to_frame("count"))


# ------------------------------------------------------------
# 2. How many labels does each subject have?
# ------------------------------------------------------------

subject_label_counts = (
    train.groupby("subject")["label"]
    .nunique()
)

print("\n=== LABEL DIVERSITY PER SUBJECT ===")

print(
    "Subjects with exactly 1 label:",
    (subject_label_counts == 1).sum()
)

print(
    "Subjects with 2+ labels:",
    (subject_label_counts >= 2).sum()
)

print(
    "Maximum labels for one subject:",
    subject_label_counts.max()
)


# ------------------------------------------------------------
# 3. Subject → dominant label
# ------------------------------------------------------------

subject_label_table = (
    train.groupby(["subject", "label"])
    .size()
    .reset_index(name="count")
)

dominant_subject_label = (
    subject_label_table
    .sort_values(["subject", "count"], ascending=[True, False])
    .drop_duplicates("subject")
)

# Merge total counts
dominant_subject_label = dominant_subject_label.merge(
    subject_counts.rename("total_count"),
    left_on="subject",
    right_index=True
)

dominant_subject_label["dominance"] = (
    dominant_subject_label["count"]
    / dominant_subject_label["total_count"]
)

print("\n=== SUBJECT LABEL DOMINANCE ===")

print(
    "Subjects with 100% same label:",
    (dominant_subject_label["dominance"] == 1.0).sum()
)

print(
    "Subjects with >=90% same label:",
    (dominant_subject_label["dominance"] >= 0.90).sum()
)

print(
    "Subjects with >=75% same label:",
    (dominant_subject_label["dominance"] >= 0.75).sum()
)

print(
    "Average dominance:",
    round(dominant_subject_label["dominance"].mean(), 4)
)


# ------------------------------------------------------------
# 4. Show ambiguous subjects
# ------------------------------------------------------------

ambiguous_subjects = (
    dominant_subject_label[
        dominant_subject_label["dominance"] < 1.0
    ]
    .sort_values(
        ["total_count", "dominance"],
        ascending=[False, True]
    )
)

print("\n=== EXAMPLES OF AMBIGUOUS SUBJECTS ===")

for subject in ambiguous_subjects["subject"].head(15):
    
    print("\nSUBJECT:", subject)
    
    display(
        train.loc[
            train["subject"] == subject,
            ["subject", "body", "channel",
             "district", "complaint_history", "label"]
        ].head(10)
    )


# ------------------------------------------------------------
# 5. Train/Test subject overlap
# ------------------------------------------------------------

train_subjects = set(train["subject"])
test_subjects = set(test["subject"])

overlap = train_subjects.intersection(test_subjects)

print("\n=== TRAIN / TEST SUBJECT OVERLAP ===")

print("Unique train subjects:", len(train_subjects))
print("Unique test subjects:", len(test_subjects))
print("Overlapping subjects:", len(overlap))

test_overlap_mask = test["subject"].isin(overlap)

print(
    "Test rows whose subject appeared in training:",
    test_overlap_mask.sum()
)

print(
    "Percentage of test rows with known subject:",
    round(test_overlap_mask.mean() * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 6. How reliable is the subject lookup on overlapping test rows?
# ------------------------------------------------------------

# Subjects that have exactly one label in training
pure_subjects = set(
    subject_label_counts[
        subject_label_counts == 1
    ].index
)

pure_overlap = overlap.intersection(pure_subjects)

print(
    "\nOverlapping subjects with exactly ONE training label:",
    len(pure_overlap)
)

pure_test_mask = test["subject"].isin(pure_overlap)

print(
    "Test rows with an unambiguous known subject:",
    pure_test_mask.sum()
)

print(
    "Percentage of test rows:",
    round(pure_test_mask.mean() * 100, 2),
    "%"
)

Total training rows: 4800
Unique subjects: 818
Repeated subjects: 275
Subjects appearing once: 543

Top 20 most frequent subjects:


,count
subject,
No response from officials,107
Nobody is listening to us,101
"Footpath encroached, pedestrians on the road",100
Problem getting worse daily,98
Long pending issue,98
Urgent problem in our area,96
Appeal to the war room,95
Please help our colony,92
Second time complaining,92



=== LABEL DIVERSITY PER SUBJECT ===
Subjects with exactly 1 label: 575
Subjects with 2+ labels: 243
Maximum labels for one subject: 24

=== SUBJECT LABEL DOMINANCE ===
Subjects with 100% same label: 575
Subjects with >=90% same label: 575
Subjects with >=75% same label: 581
Average dominance: 0.8489

=== EXAMPLES OF AMBIGUOUS SUBJECTS ===

SUBJECT: No response from officials


,subject,body,channel,district,complaint_history,label
29,No response from officials,"Sir namaskara, Lorries carrying ilelgally mine...",helpline_call,Devgiri,repeat_complaint,law_and_order|high
34,No response from officials,"Sir/Madam, The ration for our lane's ten labou...",helpline_call,Malligere,repeat_complaint,welfare_schemes|critical
38,No response from officials,"Sir ji, The co-op society has had no urea stoc...",facebook,Krishnapura,repeat_complaint,agriculture_irrigation|high
58,No response from officials,The water coming in our taps is brown and smel...,field_visit,Kotehalli,first_time,water_supply|critical
120,No response from officials,"Dear team, The government hospital pharmacy sa...",field_visit,Krishnapura,first_time,healthcare|high
257,No response from officials,"Namaskar, The gambling den behind the market i...",helpline_call,Rampura,escalated,law_and_order|routine
416,No response from officials,"Namaste, The sanction letter for the house sup...",helpline_call,Chandpur,escalated,welfare_schemes|routine
485,No response from officials,"Sir namaskara, The bridge near the lake raste ...",helpline_call,Devgiri,repeat_complaint,roads_transport|critical
527,No response from officials,"Sir ji, Dengue cases are rising in the old mar...",field_visit,Krishnapura,escalated,healthcare|high
566,No response from officials,"Sir namaste, The aspatal ward toilets are in b...",janata_darshan,Krishnapura,first_time,healthcare|routine



SUBJECT: Nobody is listening to us


...


SUBJECT: Complaint from residents


,subject,body,channel,district,complaint_history,label
31,Complaint from residents,"Sir/Madam, I want to add my newborn daughter's...",facebook,Chandpur,first_time,welfare_schemes|routine
107,Complaint from residents,"Sanmanya sir, The primary health centre in Mal...",facebook,Sundarnagar,escalated,healthcare|high
132,Complaint from residents,"Namaskar, Streetside vendors and parked lorrie...",helpline_call,Kotehalli,escalated,roads_transport|routine
195,Complaint from residents,"Hello, The water coming in our taps is brown a...",facebook,Rampura,escalated,water_supply|critical
276,Complaint from residents,"Dear team, The new apartment construction next...",field_visit,Krishnapura,repeat_complaint,water_supply|routine
326,Complaint from residents,"Sir/Madam, Students who applied for the post-m...",facebook,Kotehalli,repeat_complaint,education|high
373,Complaint from residents,"Pranam sir, Dengue cases are rising in the old...",helpline_call,Chandpur,repeat_complaint,healthcare|high
478,Complaint from residents,"Sir/Madam, The ration for our lane's twelve la...",helpline_call,Krishnapura,escalated,welfare_schemes|critical
636,Complaint from residents,"Namaste, What arrives in our pipes for one hou...",helpline_call,Chandpur,first_time,water_supply|routine
666,Complaint from residents,"Hello, We request a monthly specialist visit t...",field_visit,Chandpur,escalated,healthcare|routine



SUBJECT: Borewell dry, tanker not coming


,subject,body,channel,district,complaint_history,label
203,"Borewell dry, tanker not coming","Sir ji, The new apartment construction next to...",whatsapp,Malligere,first_time,water_supply|routine
223,"Borewell dry, tanker not coming","Dear team, The water coming in our taps is bro...",whatsapp,Ambalpet,repeat_complaint,water_supply|critical
316,"Borewell dry, tanker not coming","Sir namaskara, The overhead tank motor burnt o...",helpline_call,Ambalpet,repeat_complaint,water_supply|high
374,"Borewell dry, tanker not coming","Respected sir, The new apartment construction ...",whatsapp,Ambalpet,first_time,water_supply|routine
419,"Borewell dry, tanker not coming","Namaste, Our colony in Sundarnagar has not rec...",helpline_call,Kotehalli,escalated,water_supply|high
429,"Borewell dry, tanker not coming","Namaste, The water coming in our taps is brown...",facebook,Ambalpet,repeat_complaint,water_supply|critical
482,"Borewell dry, tanker not coming","Sanmanya sir, What arrives in our pipes for on...",janata_darshan,Sundarnagar,first_time,water_supply|routine
590,"Borewell dry, tanker not coming","Sir namaskara, The overhead tank motor burnt o...",facebook,Devgiri,escalated,water_supply|high
705,"Borewell dry, tanker not coming","Respected sir, The overhead tank motor burnt o...",whatsapp,Rampura,escalated,water_supply|high
751,"Borewell dry, tanker not coming","Pranam sir, What arrives in our pipes for one ...",janata_darshan,Malligere,first_time,water_supply|routine



=== TRAIN / TEST SUBJECT OVERLAP ===
Unique train subjects: 818
Unique test subjects: 441
Overlapping subjects: 240
Test rows whose subject appeared in training: 1784
Percentage of test rows with known subject: 89.2 %

Overlapping subjects with exactly ONE training label: 70
Test rows with an unambiguous known subject: 85
Percentage of test rows: 4.25 %


In [5]:
# ============================================================
# STEP 3 — FIXED VALIDATION SPLIT + SUBJECT-ONLY BASELINE
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Create a fixed stratified split
# ------------------------------------------------------------

train_indices, valid_indices = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=train["label"]
)

print("Training rows:", len(train_indices))
print("Validation rows:", len(valid_indices))

# ------------------------------------------------------------
# 2. Extract subject
# ------------------------------------------------------------

subject_train = (
    train.loc[train_indices, "subject"]
    .fillna("")
    .astype(str)
)

subject_valid = (
    train.loc[valid_indices, "subject"]
    .fillna("")
    .astype(str)
)

y_train_subject = train.loc[
    train_indices, "label"
]

y_valid_subject = train.loc[
    valid_indices, "label"
]

# ------------------------------------------------------------
# 3. Subject TF-IDF
# ------------------------------------------------------------

subject_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),
    min_df=1,
    sublinear_tf=True
)

X_subject_train = subject_vectorizer.fit_transform(
    subject_train
)

X_subject_valid = subject_vectorizer.transform(
    subject_valid
)

print("\nSubject TF-IDF:")
print("Train shape:", X_subject_train.shape)
print("Validation shape:", X_subject_valid.shape)

# ------------------------------------------------------------
# 4. Train subject-only classifier
# ------------------------------------------------------------

subject_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

subject_clf.fit(
    X_subject_train,
    y_train_subject
)

# ------------------------------------------------------------
# 5. Predict
# ------------------------------------------------------------

subject_pred = subject_clf.predict(
    X_subject_valid
)

subject_accuracy = accuracy_score(
    y_valid_subject,
    subject_pred
)

# ------------------------------------------------------------
# 6. Result
# ------------------------------------------------------------

print(
    f"\nSUBJECT-ONLY ACCURACY: "
    f"{subject_accuracy * 100:.2f}%"
)

print("CURRENT CHAMPION: 93.85%")

print(
    f"Gap vs champion: "
    f"{(subject_accuracy - 0.9385) * 100:+.2f} percentage points"
)

Training rows: 3840
Validation rows: 960

Subject TF-IDF:
Train shape: (3840, 3302)
Validation shape: (960, 3302)

SUBJECT-ONLY ACCURACY: 32.92%
CURRENT CHAMPION: 93.85%
Gap vs champion: -60.93 percentage points


In [6]:
# ============================================================
# STEP 4 — BODY-ONLY TF-IDF BASELINE
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Body text
# ------------------------------------------------------------

body_train = (
    train.loc[train_indices, "body"]
    .fillna("")
    .astype(str)
)

body_valid = (
    train.loc[valid_indices, "body"]
    .fillna("")
    .astype(str)
)

y_train_body = train.loc[
    train_indices, "label"
]

y_valid_body = train.loc[
    valid_indices, "label"
]

# ------------------------------------------------------------
# 2. Word + character TF-IDF
# ------------------------------------------------------------

body_word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=120_000
)

body_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True,
    max_features=120_000
)

X_body_word_train = body_word_vectorizer.fit_transform(
    body_train
)

X_body_word_valid = body_word_vectorizer.transform(
    body_valid
)

X_body_char_train = body_char_vectorizer.fit_transform(
    body_train
)

X_body_char_valid = body_char_vectorizer.transform(
    body_valid
)

# ------------------------------------------------------------
# 3. Combine
# ------------------------------------------------------------

from scipy.sparse import hstack

X_body_train = hstack([
    X_body_word_train,
    X_body_char_train
])

X_body_valid = hstack([
    X_body_word_valid,
    X_body_char_valid
])

print("Body train shape:", X_body_train.shape)
print("Body validation shape:", X_body_valid.shape)

# ------------------------------------------------------------
# 4. Train classifier
# ------------------------------------------------------------

body_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

body_clf.fit(
    X_body_train,
    y_train_body
)

# ------------------------------------------------------------
# 5. Predict
# ------------------------------------------------------------

body_pred = body_clf.predict(
    X_body_valid
)

body_accuracy = accuracy_score(
    y_valid_body,
    body_pred
)

print(
    f"\nBODY-ONLY ACCURACY: "
    f"{body_accuracy * 100:.2f}%"
)

print("SUBJECT-ONLY ACCURACY: 32.92%")
print("CURRENT CHAMPION: 93.85%")

print(
    f"Body vs champion: "
    f"{(body_accuracy - 0.9385) * 100:+.2f} pp"
)

Body train shape: (3840, 24963)
Body validation shape: (960, 24963)

BODY-ONLY ACCURACY: 94.06%
SUBJECT-ONLY ACCURACY: 32.92%
CURRENT CHAMPION: 93.85%
Body vs champion: +0.21 pp


In [7]:
# ============================================================
# STEP 5 — BODY MODEL ERROR ANALYSIS
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Find incorrect predictions
# ------------------------------------------------------------

true_labels = y_valid_body.to_numpy()
pred_labels = body_pred

errors = true_labels != pred_labels

print("Validation examples:", len(true_labels))
print("Correct:", (~errors).sum())
print("Errors:", errors.sum())
print(
    "Accuracy:",
    f"{(~errors).mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 2. Split labels into category + urgency
# ------------------------------------------------------------

true_category = np.array([
    x.split("|")[0]
    for x in true_labels
])

pred_category = np.array([
    x.split("|")[0]
    for x in pred_labels
])

true_urgency = np.array([
    x.split("|")[1]
    for x in true_labels
])

pred_urgency = np.array([
    x.split("|")[1]
    for x in pred_labels
])

category_correct = true_category == pred_category
urgency_correct = true_urgency == pred_urgency

# ------------------------------------------------------------
# 3. Error types
# ------------------------------------------------------------

error_type = np.full(
    len(true_labels),
    "both_category_and_urgency"
)

error_type[
    (~category_correct) & urgency_correct
] = "category_only"

error_type[
    category_correct & (~urgency_correct)
] = "urgency_only"

error_type_series = pd.Series(
    error_type[errors],
    name="error_type"
)

print("\n=== ERROR TYPES ===")
print(error_type_series.value_counts())

print("\n=== ERROR TYPE % ===")
print(
    error_type_series
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# ------------------------------------------------------------
# 4. Category accuracy
# ------------------------------------------------------------

print("\n=== CATEGORY ACCURACY ===")
print(
    f"{category_correct.mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 5. Urgency accuracy
# ------------------------------------------------------------

print("\n=== URGENCY ACCURACY ===")
print(
    f"{urgency_correct.mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# 6. Urgency confusion matrix
# ------------------------------------------------------------

print("\n=== URGENCY CONFUSION ===")

urgency_cm = pd.crosstab(
    pd.Series(true_urgency, name="True"),
    pd.Series(pred_urgency, name="Predicted")
)

display(urgency_cm)

# ------------------------------------------------------------
# 7. Most common exact-label mistakes
# ------------------------------------------------------------

error_pairs = pd.DataFrame({
    "true": true_labels[errors],
    "predicted": pred_labels[errors]
})

print("\n=== TOP ERROR PAIRS ===")

display(
    error_pairs
    .value_counts()
    .head(20)
    .rename("count")
    .reset_index()
)

Validation examples: 960
Correct: 903
Errors: 57
Accuracy: 94.06%

=== ERROR TYPES ===
error_type
urgency_only    57
Name: count, dtype: int64

=== ERROR TYPE % ===
error_type
urgency_only    100.0
Name: proportion, dtype: float64

=== CATEGORY ACCURACY ===
100.00%

=== URGENCY ACCURACY ===
94.06%

=== URGENCY CONFUSION ===


Predicted,critical,high,routine
True,,,
critical,163,8,3
high,10,320,21
routine,4,11,420



=== TOP ERROR PAIRS ===


,true,predicted,count
0,welfare_schemes|routine,welfare_schemes|high,5
1,welfare_schemes|high,welfare_schemes|routine,5
2,education|high,education|routine,4
3,water_supply|high,water_supply|routine,4
4,healthcare|high,healthcare|critical,3
5,agriculture_irrigation|high,agriculture_irrigation|routine,3
6,electricity|high,electricity|routine,3
7,electricity|high,electricity|critical,2
8,law_and_order|routine,law_and_order|high,2
9,welfare_schemes|critical,welfare_schemes|high,2


In [8]:
# ============================================================
# STEP 6 — DEDICATED URGENCY MODEL
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Extract urgency from training/validation labels
# ------------------------------------------------------------

train_urgency = (
    train.loc[train_indices, "label"]
    .str.split("|")
    .str[1]
    .to_numpy()
)

valid_urgency = (
    train.loc[valid_indices, "label"]
    .str.split("|")
    .str[1]
    .to_numpy()
)

print("Training urgency distribution:")
print(pd.Series(train_urgency).value_counts())

print("\nValidation urgency distribution:")
print(pd.Series(valid_urgency).value_counts())

# ------------------------------------------------------------
# 2. Train dedicated urgency classifier
# ------------------------------------------------------------

urgency_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

urgency_clf.fit(
    X_body_train,
    train_urgency
)

# ------------------------------------------------------------
# 3. Predict urgency
# ------------------------------------------------------------

urgency_pred = urgency_clf.predict(
    X_body_valid
)

urgency_accuracy = accuracy_score(
    valid_urgency,
    urgency_pred
)

print(
    f"\nDEDICATED URGENCY ACCURACY: "
    f"{urgency_accuracy * 100:.2f}%"
)

print("\nClassification report:")
print(
    classification_report(
        valid_urgency,
        urgency_pred,
        digits=4
    )
)

# ------------------------------------------------------------
# 4. Compare with urgency from 24-class body model
# ------------------------------------------------------------

body_model_urgency = np.array([
    x.split("|")[1]
    for x in body_pred
])

body_urgency_accuracy = accuracy_score(
    valid_urgency,
    body_model_urgency
)

print("\n=== COMPARISON ===")

print(
    f"24-class body model urgency: "
    f"{body_urgency_accuracy * 100:.2f}%"
)

print(
    f"Dedicated urgency model:     "
    f"{urgency_accuracy * 100:.2f}%"
)

agreement = (
    body_model_urgency == urgency_pred
)

print(
    f"Model agreement:             "
    f"{agreement.mean() * 100:.2f}%"
)

print(
    f"Model disagreement:          "
    f"{(~agreement).mean() * 100:.2f}%"
)

Training urgency distribution:
routine     1740
high        1405
critical     695
Name: count, dtype: int64

Validation urgency distribution:
routine     435
high        351
critical    174
Name: count, dtype: int64

DEDICATED URGENCY ACCURACY: 94.17%

Classification report:
              precision    recall  f1-score   support

    critical     0.9162    0.9425    0.9292       174
        high     0.9520    0.9031    0.9269       351
     routine     0.9442    0.9724    0.9581       435

    accuracy                         0.9417       960
   macro avg     0.9374    0.9394    0.9381       960
weighted avg     0.9420    0.9417    0.9414       960


=== COMPARISON ===
24-class body model urgency: 94.06%
Dedicated urgency model:     94.17%
Model agreement:             97.19%
Model disagreement:          2.81%


In [9]:
# ============================================================
# STEP 7 — CATEGORY FROM 24-CLASS + URGENCY FROM SPECIALIST
# ============================================================

from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Extract category from the 24-class body model
# ------------------------------------------------------------

body_model_category = np.array([
    x.split("|")[0]
    for x in body_pred
])

# ------------------------------------------------------------
# 2. Build hybrid prediction
# ------------------------------------------------------------

hybrid_pred = np.array([
    category + "|" + urgency
    for category, urgency
    in zip(body_model_category, urgency_pred)
])

# ------------------------------------------------------------
# 3. Accuracy
# ------------------------------------------------------------

hybrid_accuracy = accuracy_score(
    y_valid_body,
    hybrid_pred
)

body_accuracy = accuracy_score(
    y_valid_body,
    body_pred
)

print("=== MODEL COMPARISON ===")

print(
    f"Body 24-class model: "
    f"{body_accuracy * 100:.2f}%"
)

print(
    f"Category + dedicated urgency: "
    f"{hybrid_accuracy * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(hybrid_accuracy - body_accuracy) * 100:+.2f} pp"
)

# ------------------------------------------------------------
# 4. How many predictions changed?
# ------------------------------------------------------------

changed = hybrid_pred != body_pred

print("\n=== CHANGES ===")
print(
    "Predictions changed:",
    changed.sum()
)

if changed.sum() > 0:

    true_arr = y_valid_body.to_numpy()

    original_correct = (
        body_pred[changed] == true_arr[changed]
    )

    hybrid_correct = (
        hybrid_pred[changed] == true_arr[changed]
    )

    print(
        "Original correct among changed:",
        original_correct.sum()
    )

    print(
        "Hybrid correct among changed:",
        hybrid_correct.sum()
    )

    print(
        "Hybrid gained:",
        (
            ~original_correct & hybrid_correct
        ).sum()
    )

    print(
        "Hybrid lost:",
        (
            original_correct & ~hybrid_correct
        ).sum()
    )

    print(
        "Both wrong:",
        (
            ~original_correct & ~hybrid_correct
        ).sum()
    )

# ------------------------------------------------------------
# 5. Show changed examples
# ------------------------------------------------------------

if changed.sum() > 0:

    changed_df = pd.DataFrame({
        "true": true_arr[changed],
        "original": body_pred[changed],
        "hybrid": hybrid_pred[changed],
        "subject": train.loc[
            valid_indices[changed],
            "subject"
        ].to_numpy()
    })

    print("\n=== CHANGED PREDICTIONS ===")
    display(changed_df)

=== MODEL COMPARISON ===
Body 24-class model: 94.06%
Category + dedicated urgency: 94.17%
Improvement: +0.10 pp

=== CHANGES ===
Predictions changed: 27
Original correct among changed: 13
Hybrid correct among changed: 14
Hybrid gained: 14
Hybrid lost: 13
Both wrong: 0

=== CHANGED PREDICTIONS ===


,true,original,hybrid,subject
0,roads_transport|routine,roads_transport|routine,roads_transport|critical,Nobody is listening to us
1,education|high,education|high,education|routine,Teacher shortage at the government school
2,healthcare|routine,healthcare|routine,healthcare|high,Dengue spreading in Gandhi colony
3,law_and_order|critical,law_and_order|high,law_and_order|critical,Chain snatching incidents in the lake road side
4,roads_transport|routine,roads_transport|critical,roads_transport|routine,Bridge condition near ward 11
5,law_and_order|critical,law_and_order|high,law_and_order|critical,Police not registering our complaint
6,law_and_order|routine,law_and_order|high,law_and_order|routine,Chain snatching incidents in the weavers’ colony
7,welfare_schemes|high,welfare_schemes|high,welfare_schemes|critical,Widow pension rejected wrongly
8,welfare_schemes|routine,welfare_schemes|critical,welfare_schemes|routine,Widow pensino rejected wrongly
9,healthcare|routine,healthcare|high,healthcare|routine,Ambulance response time complaint


In [10]:
print("=== FINAL HYBRID RESULT ===")

print(f"Body 24-class:       {body_accuracy * 100:.2f}%")
print(f"Hybrid:              {hybrid_accuracy * 100:.2f}%")
print(f"Improvement:         {(hybrid_accuracy - body_accuracy) * 100:+.2f} pp")

changed = hybrid_pred != body_pred
true_arr = y_valid_body.to_numpy()

original_correct = body_pred[changed] == true_arr[changed]
hybrid_correct = hybrid_pred[changed] == true_arr[changed]

print("\n=== CHANGED PREDICTIONS ===")
print("Changed:", changed.sum())
print("Original correct:", original_correct.sum())
print("Hybrid correct:", hybrid_correct.sum())
print("Hybrid gained:", (~original_correct & hybrid_correct).sum())
print("Hybrid lost:", (original_correct & ~hybrid_correct).sum())
print("Both wrong:", (~original_correct & ~hybrid_correct).sum())

=== FINAL HYBRID RESULT ===
Body 24-class:       94.06%
Hybrid:              94.17%
Improvement:         +0.10 pp

=== CHANGED PREDICTIONS ===
Changed: 27
Original correct: 13
Hybrid correct: 14
Hybrid gained: 14
Hybrid lost: 13
Both wrong: 0


In [11]:
# ============================================================
# STEP 8 — CONFIDENCE-AWARE URGENCY ENSEMBLE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Get decision scores
# ------------------------------------------------------------

scores_24 = body_clf.decision_function(X_body_valid)
scores_urgency = urgency_clf.decision_function(X_body_valid)

classes_24 = body_clf.classes_
classes_urgency = urgency_clf.classes_

print("24-class score shape:", scores_24.shape)
print("Urgency score shape:", scores_urgency.shape)

# ------------------------------------------------------------
# 2. Convert 24-class scores → urgency scores
# ------------------------------------------------------------

urgency_names = ["critical", "high", "routine"]

urgency_24_scores = np.zeros(
    (X_body_valid.shape[0], 3)
)

for i, urgency in enumerate(urgency_names):

    mask = np.array([
        label.split("|")[1] == urgency
        for label in classes_24
    ])

    urgency_24_scores[:, i] = scores_24[:, mask].max(axis=1)

# ------------------------------------------------------------
# 3. Get predicted urgency from each model
# ------------------------------------------------------------

urgency_24_idx = urgency_24_scores.argmax(axis=1)

urgency_specialist_idx = scores_urgency.argmax(axis=1)

urgency_24_pred = np.array(
    urgency_names
)[urgency_24_idx]

urgency_specialist_pred = np.array(
    urgency_names
)[urgency_specialist_idx]

# ------------------------------------------------------------
# 4. Calculate confidence margins
# ------------------------------------------------------------

def margin_from_scores(scores):
    sorted_scores = np.sort(scores, axis=1)
    return sorted_scores[:, -1] - sorted_scores[:, -2]


margin_24 = margin_from_scores(
    urgency_24_scores
)

margin_specialist = margin_from_scores(
    scores_urgency
)

# ------------------------------------------------------------
# 5. Check disagreement
# ------------------------------------------------------------

disagree = (
    urgency_24_pred != urgency_specialist_pred
)

print("\n=== URGENCY AGREEMENT ===")

print(
    "Agreement:",
    f"{(~disagree).mean() * 100:.2f}%"
)

print(
    "Disagreement:",
    f"{disagree.mean() * 100:.2f}%"
)

print(
    "Disagreement count:",
    disagree.sum()
)

# ------------------------------------------------------------
# 6. Baseline 24-class prediction
# ------------------------------------------------------------

category_pred = np.array([
    label.split("|")[0]
    for label in body_pred
])

baseline_pred = np.array([
    cat + "|" + urg
    for cat, urg in zip(
        category_pred,
        urgency_24_pred
    )
])

baseline_accuracy = accuracy_score(
    y_valid_body,
    baseline_pred
)

print(
    "\nBaseline 24-class accuracy:",
    f"{baseline_accuracy * 100:.2f}%"
)

# ------------------------------------------------------------
# 7. Search confidence thresholds
# ------------------------------------------------------------

true_labels = y_valid_body.to_numpy()

thresholds = [
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
    0.60,
    0.75,
    1.00
]

results = []

for threshold in thresholds:

    final_urgency = urgency_24_pred.copy()

    # Switch to specialist ONLY when:
    # 1. Models disagree
    # 2. Specialist confidence is sufficiently stronger

    use_specialist = (
        disagree
        &
        (
            margin_specialist
            >
            margin_24 + threshold
        )
    )

    final_urgency[use_specialist] = (
        urgency_specialist_pred[use_specialist]
    )

    final_pred = np.array([
        cat + "|" + urg
        for cat, urg in zip(
            category_pred,
            final_urgency
        )
    ])

    acc = accuracy_score(
        true_labels,
        final_pred
    )

    results.append({
        "threshold": threshold,
        "accuracy": acc,
        "changed": int(use_specialist.sum())
    })

results_df = pd.DataFrame(results)

results_df["accuracy_pct"] = (
    results_df["accuracy"] * 100
)

print("\n=== CONFIDENCE THRESHOLD SEARCH ===")

display(
    results_df[
        [
            "threshold",
            "accuracy_pct",
            "changed"
        ]
    ]
)

# ------------------------------------------------------------
# 8. Best threshold
# ------------------------------------------------------------

best_row = results_df.loc[
    results_df["accuracy"].idxmax()
]

print("\n=== BEST RESULT ===")

print(
    f"Threshold: {best_row['threshold']}"
)

print(
    f"Accuracy: {best_row['accuracy_pct']:.2f}%"
)

print(
    f"Predictions changed: "
    f"{int(best_row['changed'])}"
)

24-class score shape: (960, 24)
Urgency score shape: (960, 3)

=== URGENCY AGREEMENT ===
Agreement: 97.19%
Disagreement: 2.81%
Disagreement count: 27

Baseline 24-class accuracy: 94.06%

=== CONFIDENCE THRESHOLD SEARCH ===


,threshold,accuracy_pct,changed
0,0.00,93.958333,11
1,0.05,94.062500,10
2,0.10,94.166667,9
3,0.15,94.062500,8
4,0.20,94.062500,8
5,0.25,94.166667,7
6,0.30,94.166667,7
7,0.40,94.062500,2
8,0.50,93.958333,1
9,0.60,93.958333,1



=== BEST RESULT ===
Threshold: 0.1
Accuracy: 94.17%
Predictions changed: 9


In [12]:
# ============================================================
# STEP 9 — HIGH vs ROUTINE SPECIALIST
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Create high/routine masks
# ------------------------------------------------------------

train_urgency_full = (
    train.loc[train_indices, "label"]
    .str.split("|")
    .str[1]
    .to_numpy()
)

valid_urgency_full = (
    train.loc[valid_indices, "label"]
    .str.split("|")
    .str[1]
    .to_numpy()
)

hr_train_mask = np.isin(
    train_urgency_full,
    ["high", "routine"]
)

hr_valid_mask = np.isin(
    valid_urgency_full,
    ["high", "routine"]
)

X_train_hr = X_body_train[hr_train_mask]
X_valid_hr = X_body_valid[hr_valid_mask]

y_train_hr = train_urgency_full[hr_train_mask]
y_valid_hr = valid_urgency_full[hr_valid_mask]

print("Training samples:", X_train_hr.shape[0])
print("Validation samples:", X_valid_hr.shape[0])

print("\nTraining distribution:")
print(pd.Series(y_train_hr).value_counts())

# ------------------------------------------------------------
# 2. Train specialist
# ------------------------------------------------------------

hr_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

hr_clf.fit(
    X_train_hr,
    y_train_hr
)

# ------------------------------------------------------------
# 3. Predict
# ------------------------------------------------------------

hr_pred = hr_clf.predict(
    X_valid_hr
)

hr_accuracy = accuracy_score(
    y_valid_hr,
    hr_pred
)

print(
    f"\nHIGH vs ROUTINE ACCURACY: "
    f"{hr_accuracy * 100:.2f}%"
)

print("\nClassification report:")

print(
    classification_report(
        y_valid_hr,
        hr_pred,
        digits=4
    )
)

Training samples: 3145
Validation samples: 786

Training distribution:
routine    1740
high       1405
Name: count, dtype: int64

HIGH vs ROUTINE ACCURACY: 95.42%

Classification report:
              precision    recall  f1-score   support

        high     0.9674    0.9288    0.9477       351
     routine     0.9443    0.9747    0.9593       435

    accuracy                         0.9542       786
   macro avg     0.9558    0.9517    0.9535       786
weighted avg     0.9546    0.9542    0.9541       786



In [13]:
# ============================================================
# STEP 10 — HIGH/ROUTINE SPECIALIST HYBRID
# ============================================================

# Get 24-class model urgency
body_urgency = np.array([
    x.split("|")[1]
    for x in body_pred
])

# Get categories from 24-class model
body_category = np.array([
    x.split("|")[0]
    for x in body_pred
])

# Start with original prediction
hr_hybrid_pred = body_pred.copy()

# Only replace urgency for validation examples that are
# actually high/routine
hr_indices = np.where(hr_valid_mask)[0]

# Replace their urgency with specialist prediction
for local_idx, original_idx in enumerate(hr_indices):

    hr_hybrid_pred[original_idx] = (
        body_category[original_idx]
        + "|"
        + hr_pred[local_idx]
    )

# Evaluate
hr_hybrid_accuracy = accuracy_score(
    y_valid_body,
    hr_hybrid_pred
)

print("=== HIGH/ROUTINE HYBRID ===")

print(
    f"Original 24-class: "
    f"{body_accuracy * 100:.2f}%"
)

print(
    f"HR specialist hybrid: "
    f"{hr_hybrid_accuracy * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(hr_hybrid_accuracy - body_accuracy) * 100:+.2f} pp"
)

# ------------------------------------------------------------
# Changed predictions
# ------------------------------------------------------------

changed = hr_hybrid_pred != body_pred
true_arr = y_valid_body.to_numpy()

original_correct = (
    body_pred[changed] == true_arr[changed]
)

hybrid_correct = (
    hr_hybrid_pred[changed] == true_arr[changed]
)

print("\n=== CHANGES ===")

print("Predictions changed:", changed.sum())

print(
    "Original correct:",
    original_correct.sum()
)

print(
    "Hybrid correct:",
    hybrid_correct.sum()
)

print(
    "Hybrid gained:",
    (
        ~original_correct & hybrid_correct
    ).sum()
)

print(
    "Hybrid lost:",
    (
        original_correct & ~hybrid_correct
    ).sum()
)

print(
    "Both wrong:",
    (
        ~original_correct & ~hybrid_correct
    ).sum()
)

=== HIGH/ROUTINE HYBRID ===
Original 24-class: 94.06%
HR specialist hybrid: 95.10%
Improvement: +1.04 pp

=== CHANGES ===
Predictions changed: 28
Original correct: 9
Hybrid correct: 19
Hybrid gained: 19
Hybrid lost: 9
Both wrong: 0


In [14]:
# ============================================================
# STEP 11 — SECOND-SPLIT STRESS TEST
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. NEW, INDEPENDENT STRATIFIED SPLIT
# ------------------------------------------------------------

stress_train_idx, stress_valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=123,
    stratify=train["label"]
)

print("Stress training rows:", len(stress_train_idx))
print("Stress validation rows:", len(stress_valid_idx))

# ------------------------------------------------------------
# 2. BODY TEXT
# ------------------------------------------------------------

stress_train_text = (
    train.loc[stress_train_idx, "body"]
    .fillna("")
    .astype(str)
)

stress_valid_text = (
    train.loc[stress_valid_idx, "body"]
    .fillna("")
    .astype(str)
)

stress_y_train = train.loc[
    stress_train_idx, "label"
].to_numpy()

stress_y_valid = train.loc[
    stress_valid_idx, "label"
].to_numpy()

# ------------------------------------------------------------
# 3. FIT WORD TF-IDF
# ------------------------------------------------------------

stress_word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    max_features=120_000
)

X_stress_word_train = (
    stress_word_vectorizer.fit_transform(
        stress_train_text
    )
)

X_stress_word_valid = (
    stress_word_vectorizer.transform(
        stress_valid_text
    )
)

# ------------------------------------------------------------
# 4. FIT CHARACTER TF-IDF
# ------------------------------------------------------------

stress_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True,
    max_features=120_000
)

X_stress_char_train = (
    stress_char_vectorizer.fit_transform(
        stress_train_text
    )
)

X_stress_char_valid = (
    stress_char_vectorizer.transform(
        stress_valid_text
    )
)

# ------------------------------------------------------------
# 5. COMBINE
# ------------------------------------------------------------

from scipy.sparse import hstack

X_stress_train = hstack([
    X_stress_word_train,
    X_stress_char_train
])

X_stress_valid = hstack([
    X_stress_word_valid,
    X_stress_char_valid
])

print("\nFeature shapes:")
print("Train:", X_stress_train.shape)
print("Valid:", X_stress_valid.shape)

# ------------------------------------------------------------
# 6. 24-CLASS MODEL
# ------------------------------------------------------------

stress_24_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

stress_24_clf.fit(
    X_stress_train,
    stress_y_train
)

stress_24_pred = stress_24_clf.predict(
    X_stress_valid
)

stress_24_accuracy = accuracy_score(
    stress_y_valid,
    stress_24_pred
)

print(
    f"\n24-class accuracy: "
    f"{stress_24_accuracy * 100:.2f}%"
)

# ------------------------------------------------------------
# 7. EXTRACT URGENCY
# ------------------------------------------------------------

stress_train_urgency = np.array([
    x.split("|")[1]
    for x in stress_y_train
])

stress_valid_urgency = np.array([
    x.split("|")[1]
    for x in stress_y_valid
])

# ------------------------------------------------------------
# 8. HIGH/ROUTINE SPECIALIST
# ------------------------------------------------------------

stress_hr_train_mask = np.isin(
    stress_train_urgency,
    ["high", "routine"]
)

stress_hr_valid_mask = np.isin(
    stress_valid_urgency,
    ["high", "routine"]
)

stress_hr_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

stress_hr_clf.fit(
    X_stress_train[stress_hr_train_mask],
    stress_train_urgency[stress_hr_train_mask]
)

stress_hr_pred = stress_hr_clf.predict(
    X_stress_valid[stress_hr_valid_mask]
)

stress_hr_accuracy = accuracy_score(
    stress_valid_urgency[stress_hr_valid_mask],
    stress_hr_pred
)

print(
    f"High vs routine specialist: "
    f"{stress_hr_accuracy * 100:.2f}%"
)

# ------------------------------------------------------------
# 9. BUILD HIGH/ROUTINE HYBRID
# ------------------------------------------------------------

stress_category = np.array([
    x.split("|")[0]
    for x in stress_24_pred
])

stress_urgency_24 = np.array([
    x.split("|")[1]
    for x in stress_24_pred
])

stress_hybrid_pred = stress_24_pred.copy()

stress_hr_indices = np.where(
    stress_hr_valid_mask
)[0]

for local_idx, original_idx in enumerate(
    stress_hr_indices
):
    stress_hybrid_pred[original_idx] = (
        stress_category[original_idx]
        + "|"
        + stress_hr_pred[local_idx]
    )

stress_hybrid_accuracy = accuracy_score(
    stress_y_valid,
    stress_hybrid_pred
)

# ------------------------------------------------------------
# 10. FINAL COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("SECOND-SPLIT STRESS TEST")
print("=" * 55)

print(
    f"24-class model:       "
    f"{stress_24_accuracy * 100:.2f}%"
)

print(
    f"HR hybrid:            "
    f"{stress_hybrid_accuracy * 100:.2f}%"
)

print(
    f"Improvement:          "
    f"{(stress_hybrid_accuracy - stress_24_accuracy) * 100:+.2f} pp"
)

print("\nOriginal split champion: 95.10%")

Stress training rows: 3840
Stress validation rows: 960

Feature shapes:
Train: (3840, 24979)
Valid: (960, 24979)

24-class accuracy: 95.21%
High vs routine specialist: 97.20%

SECOND-SPLIT STRESS TEST
24-class model:       95.21%
HR hybrid:            96.35%
Improvement:          +1.15 pp

Original split champion: 95.10%


In [15]:
# ============================================================
# STEP 12 — CATEGORY-SPECIFIC HIGH/ROUTINE SAMPLE COUNTS
# ============================================================

import pandas as pd

# Use the ORIGINAL training data
train_category = (
    train.loc[train_indices, "label"]
    .str.split("|")
    .str[0]
)

train_urgency = (
    train.loc[train_indices, "label"]
    .str.split("|")
    .str[1]
)

hr_mask = train_urgency.isin(
    ["high", "routine"]
)

category_hr_counts = pd.crosstab(
    train_category[hr_mask],
    train_urgency[hr_mask]
)

category_hr_counts["total"] = (
    category_hr_counts["high"]
    + category_hr_counts["routine"]
)

print("=== HIGH vs ROUTINE COUNTS BY CATEGORY ===")
display(
    category_hr_counts.sort_values(
        "total",
        ascending=False
    )
)

print("\nMinimum category H/R samples:")
print(category_hr_counts["total"].min())

print("\nMaximum category H/R samples:")
print(category_hr_counts["total"].max())

=== HIGH vs ROUTINE COUNTS BY CATEGORY ===


label,high,routine,total
label,,,
roads_transport,230,275,505
water_supply,194,235,429
welfare_schemes,214,207,421
electricity,157,246,403
healthcare,176,208,384
agriculture_irrigation,143,206,349
law_and_order,155,193,348
education,136,170,306



Minimum category H/R samples:
306

Maximum category H/R samples:
505


In [16]:
# ============================================================
# STEP 13 — CATEGORY-SPECIFIC HIGH/ROUTINE SPECIALISTS
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Extract category + urgency
# ------------------------------------------------------------

train_labels = train.loc[
    train_indices, "label"
].to_numpy()

valid_labels = train.loc[
    valid_indices, "label"
].to_numpy()

train_categories = np.array([
    x.split("|")[0]
    for x in train_labels
])

train_urgencies = np.array([
    x.split("|")[1]
    for x in train_labels
])

valid_true_categories = np.array([
    x.split("|")[0]
    for x in valid_labels
])

valid_true_urgencies = np.array([
    x.split("|")[1]
    for x in valid_labels
])

# ------------------------------------------------------------
# 2. Categories
# ------------------------------------------------------------

categories = sorted(
    train["label"]
    .str.split("|")
    .str[0]
    .unique()
)

print("Categories:", categories)

# ------------------------------------------------------------
# 3. Train one H/R specialist per category
# ------------------------------------------------------------

category_specialists = {}

print("\n=== TRAINING CATEGORY SPECIALISTS ===")

for category in categories:

    # Training examples for this category
    category_mask = (
        train_categories == category
    )

    hr_mask = (
        category_mask
        &
        np.isin(
            train_urgencies,
            ["high", "routine"]
        )
    )

    X_cat_train = X_body_train[hr_mask]
    y_cat_train = train_urgencies[hr_mask]

    clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    clf.fit(
        X_cat_train,
        y_cat_train
    )

    category_specialists[category] = clf

    print(
        f"{category:25s} "
        f"n={len(y_cat_train):3d} "
        f"high={(y_cat_train == 'high').sum():3d} "
        f"routine={(y_cat_train == 'routine').sum():3d}"
    )

# ------------------------------------------------------------
# 4. Start from 24-class predictions
# ------------------------------------------------------------

category_hr_pred = body_pred.copy()

body_pred_categories = np.array([
    x.split("|")[0]
    for x in body_pred
])

body_pred_urgencies = np.array([
    x.split("|")[1]
    for x in body_pred
])

# ------------------------------------------------------------
# 5. Apply the appropriate specialist
# ------------------------------------------------------------

total_changed = 0

for category in categories:

    # Validation rows where the 24-class model
    # predicts this category AND H/R
    mask = (
        (body_pred_categories == category)
        &
        np.isin(
            body_pred_urgencies,
            ["high", "routine"]
        )
    )

    indices = np.where(mask)[0]

    if len(indices) == 0:
        continue

    specialist = category_specialists[category]

    specialist_pred = specialist.predict(
        X_body_valid[indices]
    )

    for local_idx, original_idx in enumerate(indices):

        category_hr_pred[original_idx] = (
            category
            + "|"
            + specialist_pred[local_idx]
        )

    total_changed += len(indices)

# ------------------------------------------------------------
# 6. Evaluate
# ------------------------------------------------------------

original_accuracy = accuracy_score(
    valid_labels,
    body_pred
)

category_hr_accuracy = accuracy_score(
    valid_labels,
    category_hr_pred
)

print("\n" + "=" * 60)
print("CATEGORY-SPECIFIC H/R RESULT")
print("=" * 60)

print(
    f"Original 24-class: "
    f"{original_accuracy * 100:.2f}%"
)

print(
    f"Category H/R hybrid: "
    f"{category_hr_accuracy * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(category_hr_accuracy - original_accuracy) * 100:+.2f} pp"
)

# ------------------------------------------------------------
# 7. Compare against GLOBAL H/R specialist
# ------------------------------------------------------------

print(
    f"\nGlobal H/R hybrid: "
    f"{95.10:.2f}%"
)

print(
    f"Category-specific vs global: "
    f"{(category_hr_accuracy - 0.9510) * 100:+.2f} pp"
)

# ------------------------------------------------------------
# 8. Changed predictions
# ------------------------------------------------------------

changed = (
    category_hr_pred != body_pred
)

true_arr = valid_labels

original_correct = (
    body_pred[changed] == true_arr[changed]
)

new_correct = (
    category_hr_pred[changed]
    == true_arr[changed]
)

print("\n=== CHANGES ===")

print(
    "Predictions changed:",
    changed.sum()
)

print(
    "Original correct:",
    original_correct.sum()
)

print(
    "New model correct:",
    new_correct.sum()
)

print(
    "Gained:",
    (
        ~original_correct & new_correct
    ).sum()
)

print(
    "Lost:",
    (
        original_correct & ~new_correct
    ).sum()
)

print(
    "Both wrong:",
    (
        ~original_correct & ~new_correct
    ).sum()
)

Categories: ['agriculture_irrigation', 'education', 'electricity', 'healthcare', 'law_and_order', 'roads_transport', 'water_supply', 'welfare_schemes']

=== TRAINING CATEGORY SPECIALISTS ===
agriculture_irrigation    n=349 high=143 routine=206
education                 n=306 high=136 routine=170
electricity               n=403 high=157 routine=246
healthcare                n=384 high=176 routine=208
law_and_order             n=348 high=155 routine=193
roads_transport           n=505 high=230 routine=275
water_supply              n=429 high=194 routine=235
welfare_schemes           n=421 high=214 routine=207

CATEGORY-SPECIFIC H/R RESULT
Original 24-class: 94.06%
Category H/R hybrid: 94.27%
Improvement: +0.21 pp

Global H/R hybrid: 95.10%
Category-specific vs global: -0.83 pp

=== CHANGES ===
Predictions changed: 10
Original correct: 4
New model correct: 6
Gained: 6
Lost: 4
Both wrong: 0


In [17]:
# ============================================================
# STEP 14 — 5-FOLD OUT-OF-FOLD VALIDATION
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

from scipy.sparse import hstack

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_SPLITS = 5
RANDOM_STATE = 42

texts = (
    train["body"]
    .fillna("")
    .astype(str)
    .to_numpy()
)

labels = train["label"].to_numpy()

# OOF predictions
oof_pred = np.empty(len(train), dtype=object)

# Track fold results
fold_results = []

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# Fold loop
# ------------------------------------------------------------

for fold, (fold_train_idx, fold_valid_idx) in enumerate(
    skf.split(texts, labels),
    start=1
):

    print("\n" + "=" * 65)
    print(f"FOLD {fold}/{N_SPLITS}")
    print("=" * 65)

    fold_train_text = texts[fold_train_idx]
    fold_valid_text = texts[fold_valid_idx]

    fold_y_train = labels[fold_train_idx]
    fold_y_valid = labels[fold_valid_idx]

    # --------------------------------------------------------
    # WORD TF-IDF
    # --------------------------------------------------------

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
        max_features=120_000
    )

    X_word_train = word_vectorizer.fit_transform(
        fold_train_text
    )

    X_word_valid = word_vectorizer.transform(
        fold_valid_text
    )

    # --------------------------------------------------------
    # CHARACTER TF-IDF
    # --------------------------------------------------------

    char_vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=120_000
    )

    X_char_train = char_vectorizer.fit_transform(
        fold_train_text
    )

    X_char_valid = char_vectorizer.transform(
        fold_valid_text
    )

    # --------------------------------------------------------
    # COMBINE FEATURES
    # --------------------------------------------------------

    X_fold_train = hstack([
        X_word_train,
        X_char_train
    ])

    X_fold_valid = hstack([
        X_word_valid,
        X_char_valid
    ])

    print(
        "Feature shape:",
        X_fold_train.shape
    )

    # --------------------------------------------------------
    # 24-CLASS MODEL
    # --------------------------------------------------------

    base_clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    base_clf.fit(
        X_fold_train,
        fold_y_train
    )

    base_pred = base_clf.predict(
        X_fold_valid
    )

    base_accuracy = accuracy_score(
        fold_y_valid,
        base_pred
    )

    # --------------------------------------------------------
    # HIGH / ROUTINE SPECIALIST
    # --------------------------------------------------------

    train_urgency = np.array([
        x.split("|")[1]
        for x in fold_y_train
    ])

    hr_train_mask = np.isin(
        train_urgency,
        ["high", "routine"]
    )

    hr_clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    hr_clf.fit(
        X_fold_train[hr_train_mask],
        train_urgency[hr_train_mask]
    )

    # --------------------------------------------------------
    # Apply specialist ONLY where base predicts H/R
    # --------------------------------------------------------

    base_category = np.array([
        x.split("|")[0]
        for x in base_pred
    ])

    base_urgency = np.array([
        x.split("|")[1]
        for x in base_pred
    ])

    final_pred = base_pred.copy()

    hr_valid_mask = np.isin(
        base_urgency,
        ["high", "routine"]
    )

    hr_indices = np.where(
        hr_valid_mask
    )[0]

    if len(hr_indices) > 0:

        specialist_pred = hr_clf.predict(
            X_fold_valid[hr_indices]
        )

        for local_idx, original_idx in enumerate(
            hr_indices
        ):
            final_pred[original_idx] = (
                base_category[original_idx]
                + "|"
                + specialist_pred[local_idx]
            )

    # --------------------------------------------------------
    # Fold scores
    # --------------------------------------------------------

    hybrid_accuracy = accuracy_score(
        fold_y_valid,
        final_pred
    )

    # Store OOF predictions
    oof_pred[fold_valid_idx] = final_pred

    fold_results.append({
        "fold": fold,
        "base_accuracy": base_accuracy,
        "hybrid_accuracy": hybrid_accuracy,
        "improvement": (
            hybrid_accuracy - base_accuracy
        )
    })

    print(
        f"24-class accuracy: "
        f"{base_accuracy * 100:.2f}%"
    )

    print(
        f"HR hybrid accuracy: "
        f"{hybrid_accuracy * 100:.2f}%"
    )

    print(
        f"Improvement: "
        f"{(hybrid_accuracy - base_accuracy) * 100:+.2f} pp"
    )

# ------------------------------------------------------------
# Overall OOF results
# ------------------------------------------------------------

results_df = pd.DataFrame(
    fold_results
)

print("\n" + "=" * 65)
print("OOF VALIDATION RESULTS")
print("=" * 65)

display(
    results_df.assign(
        base_accuracy_pct=lambda x:
            x["base_accuracy"] * 100,
        hybrid_accuracy_pct=lambda x:
            x["hybrid_accuracy"] * 100,
        improvement_pp=lambda x:
            x["improvement"] * 100
    )[
        [
            "fold",
            "base_accuracy_pct",
            "hybrid_accuracy_pct",
            "improvement_pp"
        ]
    ]
)

# ------------------------------------------------------------
# Overall OOF accuracy
# ------------------------------------------------------------

oof_accuracy = accuracy_score(
    labels,
    oof_pred
)

print(
    f"\nOverall OOF HR-hybrid accuracy: "
    f"{oof_accuracy * 100:.2f}%"
)

print(
    f"Mean fold accuracy: "
    f"{results_df['hybrid_accuracy'].mean() * 100:.2f}%"
)

print(
    f"Std fold accuracy: "
    f"{results_df['hybrid_accuracy'].std() * 100:.2f} pp"
)

print(
    f"Mean improvement over base: "
    f"{results_df['improvement'].mean() * 100:+.2f} pp"
)


FOLD 1/5
Feature shape: (3840, 25018)
24-class accuracy: 95.10%
HR hybrid accuracy: 95.00%
Improvement: -0.10 pp

FOLD 2/5
Feature shape: (3840, 25015)
24-class accuracy: 94.90%
HR hybrid accuracy: 94.58%
Improvement: -0.31 pp

FOLD 3/5
Feature shape: (3840, 24931)
24-class accuracy: 93.65%
HR hybrid accuracy: 93.33%
Improvement: -0.31 pp

FOLD 4/5
Feature shape: (3840, 24742)
24-class accuracy: 94.90%
HR hybrid accuracy: 94.58%
Improvement: -0.31 pp

FOLD 5/5
Feature shape: (3840, 24821)
24-class accuracy: 93.65%
HR hybrid accuracy: 93.44%
Improvement: -0.21 pp

OOF VALIDATION RESULTS


,fold,base_accuracy_pct,hybrid_accuracy_pct,improvement_pp
0,1,95.104167,95.000000,-0.104167
1,2,94.895833,94.583333,-0.312500
2,3,93.645833,93.333333,-0.312500
3,4,94.895833,94.583333,-0.312500
4,5,93.645833,93.437500,-0.208333



Overall OOF HR-hybrid accuracy: 94.19%
Mean fold accuracy: 94.19%
Std fold accuracy: 0.75 pp
Mean improvement over base: -0.25 pp


In [18]:
# ============================================================
# STEP 15 — SUBJECT LABEL CONSISTENCY
# ============================================================

subject_stats = (
    train.groupby("subject")
    .agg(
        examples=("label", "size"),
        unique_labels=("label", "nunique"),
        unique_categories=(
            "label",
            lambda x: x.str.split("|").str[0].nunique()
        ),
        unique_urgencies=(
            "label",
            lambda x: x.str.split("|").str[1].nunique()
        )
    )
    .reset_index()
)

print("=== SUBJECT CONSISTENCY ===")

print(
    "Unique subjects:",
    len(subject_stats)
)

print(
    "\nSubject frequency:"
)

display(
    subject_stats["examples"]
    .value_counts()
    .sort_index()
    .head(20)
)

print(
    "\nSubjects with exactly one label:",
    (
        subject_stats["unique_labels"] == 1
    ).sum()
)

print(
    "Subjects with multiple labels:",
    (
        subject_stats["unique_labels"] > 1
    ).sum()
)

print(
    "Subjects with exactly one category:",
    (
        subject_stats["unique_categories"] == 1
    ).sum()
)

print(
    "Subjects with exactly one urgency:",
    (
        subject_stats["unique_urgencies"] == 1
    ).sum()
)

print("\n=== MOST COMMON SUBJECTS ===")

display(
    subject_stats.sort_values(
        "examples",
        ascending=False
    ).head(30)
)

=== SUBJECT CONSISTENCY ===
Unique subjects: 818

Subject frequency:


examples
1     543
2     106
3      23
4      17
5       4
6       7
7      15
8      11
9      13
10     10
11     11
12      5
13      5
14      4
15      2
18      1
20      1
39      1
43      1
56      1
Name: count, dtype: int64


Subjects with exactly one label: 575
Subjects with multiple labels: 243
Subjects with exactly one category: 759
Subjects with exactly one urgency: 591

=== MOST COMMON SUBJECTS ===


,subject,examples,unique_labels,unique_categories,unique_urgencies
443,No response from officials,107,24,8,3
461,Nobody is listening to us,101,22,8,3
200,"Footpath encroached, pedestrians on the road",100,3,1,3
557,Problem getting worse daily,98,23,8,3
262,Long pending issue,98,24,8,3
754,Urgent problem in our area,96,24,8,3
16,Appeal to the war room,95,23,8,3
681,Second time complaining,92,22,8,3
495,Please help our colony,92,22,8,3
251,Kindly look into this,90,22,8,3


In [19]:
# ============================================================
# STEP 16 — HIGH-CONFIDENCE SUBJECT CATEGORY PRIOR
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Training subject → category statistics
# ------------------------------------------------------------

train_subject_stats = (
    train.loc[train_indices]
    .assign(
        category=lambda df:
            df["label"].str.split("|").str[0]
    )
    .groupby("subject")
    .agg(
        examples=("label", "size"),
        unique_categories=("category", "nunique")
    )
    .reset_index()
)

# Keep only subjects with exactly one category
unambiguous_subjects = train_subject_stats[
    train_subject_stats["unique_categories"] == 1
].copy()

# Map subject → category
subject_to_category = {}

for _, row in unambiguous_subjects.iterrows():

    subject = row["subject"]

    category = (
        train.loc[
            train_indices
        ]
        .loc[
            train.loc[
                train_indices,
                "subject"
            ] == subject,
            "label"
        ]
        .str.split("|")
        .str[0]
        .iloc[0]
    )

    subject_to_category[subject] = category

# ------------------------------------------------------------
# 2. Validation subjects
# ------------------------------------------------------------

valid_subjects = (
    train.loc[
        valid_indices,
        "subject"
    ]
    .fillna("")
    .astype(str)
    .to_numpy()
)

valid_true = train.loc[
    valid_indices,
    "label"
].to_numpy()

valid_true_category = np.array([
    x.split("|")[0]
    for x in valid_true
])

# Category predicted by body model
body_category = np.array([
    x.split("|")[0]
    for x in body_pred
])

body_urgency = np.array([
    x.split("|")[1]
    for x in body_pred
])

# ------------------------------------------------------------
# 3. Test different minimum subject frequencies
# ------------------------------------------------------------

thresholds = [5, 10, 20, 30, 50]

results = []

for min_count in thresholds:

    # Subjects satisfying:
    #   exactly one category
    #   enough training examples

    eligible_stats = unambiguous_subjects[
        unambiguous_subjects["examples"]
        >= min_count
    ]

    eligible_subject_set = set(
        eligible_stats["subject"]
    )

    final_category = body_category.copy()

    used = np.zeros(
        len(valid_subjects),
        dtype=bool
    )

    for i, subject in enumerate(
        valid_subjects
    ):

        if subject in eligible_subject_set:

            final_category[i] = (
                subject_to_category[subject]
            )

            used[i] = True

    final_pred = np.array([
        category + "|" + urgency
        for category, urgency
        in zip(
            final_category,
            body_urgency
        )
    ])

    accuracy = accuracy_score(
        valid_true,
        final_pred
    )

    category_accuracy = accuracy_score(
        valid_true_category,
        final_category
    )

    results.append({
        "min_examples": min_count,
        "accuracy": accuracy,
        "accuracy_pct": accuracy * 100,
        "category_accuracy": category_accuracy,
        "category_accuracy_pct":
            category_accuracy * 100,
        "overrides": int(used.sum())
    })

results_df = pd.DataFrame(results)

print("=== SUBJECT CATEGORY PRIOR ===")

display(
    results_df[
        [
            "min_examples",
            "accuracy_pct",
            "category_accuracy_pct",
            "overrides"
        ]
    ]
)

print(
    "\nBody-only baseline:",
    f"{accuracy_score(valid_true, body_pred) * 100:.2f}%"
)

print(
    "Current global H/R hybrid:",
    "95.10%"
)

=== SUBJECT CATEGORY PRIOR ===


,min_examples,accuracy_pct,category_accuracy_pct,overrides
0,5,94.0625,100.0,537
1,10,94.0625,100.0,449
2,20,94.0625,100.0,404
3,30,94.0625,100.0,404
4,50,94.0625,100.0,285



Body-only baseline: 94.06%
Current global H/R hybrid: 95.10%


In [20]:
print("=== SUBJECT CATEGORY PRIOR RESULTS ===")

display(
    results_df[
        [
            "min_examples",
            "accuracy_pct",
            "category_accuracy_pct",
            "overrides"
        ]
    ]
)

print(
    "\nBody-only baseline:",
    f"{accuracy_score(valid_true, body_pred) * 100:.2f}%"
)

print(
    "Current global H/R hybrid:",
    "95.10%"
)

=== SUBJECT CATEGORY PRIOR RESULTS ===


,min_examples,accuracy_pct,category_accuracy_pct,overrides
0,5,94.0625,100.0,537
1,10,94.0625,100.0,449
2,20,94.0625,100.0,404
3,30,94.0625,100.0,404
4,50,94.0625,100.0,285



Body-only baseline: 94.06%
Current global H/R hybrid: 95.10%


In [21]:
import os

print("=== KAGGLE INPUT ===")
print(os.listdir("/kaggle/input"))

print("\n=== BGE-M3 PATHS ===")

for root, dirs, files in os.walk("/kaggle/input"):
    if "bge-m3" in root.lower():
        print(root)

=== KAGGLE INPUT ===
['competitions', 'models']

=== BGE-M3 PATHS ===
/kaggle/input/models/yethukmutt/bge-m3
/kaggle/input/models/yethukmutt/bge-m3/transformers
/kaggle/input/models/yethukmutt/bge-m3/transformers/m3
/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1
/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3
/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3/1_Pooling
/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3/onnx
/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3/imgs


In [22]:
# ============================================================
# STEP 18 — LOAD LOCAL BGE-M3
# ============================================================

from sentence_transformers import SentenceTransformer

MODEL_PATH = (
    "/kaggle/input/models/yethukmutt/"
    "bge-m3/transformers/m3/1/bge-m3"
)

print("Loading BGE-M3...")

bge_encoder = SentenceTransformer(
    MODEL_PATH,
    device="cuda" if __import__("torch").cuda.is_available() else "cpu"
)

print("Model loaded successfully!")

print(
    "Embedding dimension:",
    bge_encoder.get_embedding_dimension()
)

Loading BGE-M3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded successfully!
Embedding dimension: 1024


In [23]:
import numpy as np

sample_texts = (
    train.loc[train_indices, "subject"]
    .fillna("")
    .astype(str)
    + " [SEP] "
    + train.loc[train_indices, "body"]
    .fillna("")
    .astype(str)
).iloc[:5].tolist()

sample_embeddings = bge_encoder.encode(
    sample_texts,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

print("Embedding shape:", sample_embeddings.shape)
print("Data type:", sample_embeddings.dtype)
print("Contains NaN:", np.isnan(sample_embeddings).any())

print("\nFirst embedding (first 10 values):")
print(sample_embeddings[0][:10])

print("\nEmbedding norms:")
print(np.linalg.norm(sample_embeddings, axis=1))

Embedding shape: (5, 1024)
Data type: float32
Contains NaN: False

First embedding (first 10 values):
[-0.05275011 -0.00787905 -0.03103989  0.00599786 -0.0007723  -0.06152196
 -0.01662135  0.01831061 -0.00535992 -0.01301938]

Embedding norms:
[1.        1.        1.0000001 1.        1.       ]


In [24]:
# ============================================================
# STEP 19 — FULL BGE-M3 EMBEDDINGS
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Build texts
# ------------------------------------------------------------

bge_train_texts = (
    train.loc[train_indices, "subject"]
    .fillna("")
    .astype(str)
    + " [SEP] "
    + train.loc[train_indices, "body"]
    .fillna("")
    .astype(str)
).tolist()

bge_valid_texts = (
    train.loc[valid_indices, "subject"]
    .fillna("")
    .astype(str)
    + " [SEP] "
    + train.loc[valid_indices, "body"]
    .fillna("")
    .astype(str)
).tolist()

print("Training texts:", len(bge_train_texts))
print("Validation texts:", len(bge_valid_texts))

# ------------------------------------------------------------
# Encode training
# ------------------------------------------------------------

print("\nEncoding training data...")

bge_train_embeddings = bge_encoder.encode(
    bge_train_texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

# ------------------------------------------------------------
# Encode validation
# ------------------------------------------------------------

print("\nEncoding validation data...")

bge_valid_embeddings = bge_encoder.encode(
    bge_valid_texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n=== BGE-M3 EMBEDDINGS ===")

print(
    "Train shape:",
    bge_train_embeddings.shape
)

print(
    "Validation shape:",
    bge_valid_embeddings.shape
)

print(
    "Train dtype:",
    bge_train_embeddings.dtype
)

print(
    "Validation dtype:",
    bge_valid_embeddings.dtype
)

print(
    "Train NaN:",
    np.isnan(bge_train_embeddings).any()
)

print(
    "Validation NaN:",
    np.isnan(bge_valid_embeddings).any()
)

print(
    "Approx memory:",
    (
        bge_train_embeddings.nbytes
        + bge_valid_embeddings.nbytes
    ) / (1024 ** 2),
    "MB"
)

Training texts: 3840
Validation texts: 960

Encoding training data...


Batches:   0%|          | 0/120 [00:00<?, ?it/s]


Encoding validation data...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]


=== BGE-M3 EMBEDDINGS ===
Train shape: (3840, 1024)
Validation shape: (960, 1024)
Train dtype: float32
Validation dtype: float32
Train NaN: False
Validation NaN: False
Approx memory: 18.75 MB


In [26]:
# ============================================================
# STEP 20 — BGE-M3 ONLY CLASSIFIER
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

bge_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

print("Training BGE-M3 LinearSVC...")

bge_clf.fit(
    bge_train_embeddings,
    train.loc[train_indices, "label"].to_numpy()
)

bge_pred = bge_clf.predict(
    bge_valid_embeddings
)

bge_accuracy = accuracy_score(
    train.loc[valid_indices, "label"].to_numpy(),
    bge_pred
)

print(
    f"\nBGE-M3 accuracy: "
    f"{bge_accuracy * 100:.2f}%"
)

print("\nClassification report:")
print(
    classification_report(
        train.loc[valid_indices, "label"].to_numpy(),
        bge_pred,
        digits=4
    )
)

print("\n=== BASELINE COMPARISON ===")

print(
    f"TF-IDF 24-class: "
    f"{94.06:.2f}%"
)

print(
    f"BGE-M3:          "
    f"{bge_accuracy * 100:.2f}%"
)

print(
    f"Difference:      "
    f"{(bge_accuracy - 0.9406) * 100:+.2f} pp"
)

Training BGE-M3 LinearSVC...

BGE-M3 accuracy: 98.65%

Classification report:
                                 precision    recall  f1-score   support

agriculture_irrigation|critical     1.0000    1.0000    1.0000        19
    agriculture_irrigation|high     1.0000    0.9167    0.9565        36
 agriculture_irrigation|routine     0.9455    1.0000    0.9720        52
             education|critical     1.0000    1.0000    1.0000        18
                 education|high     1.0000    0.9706    0.9851        34
              education|routine     0.9767    1.0000    0.9882        42
           electricity|critical     1.0000    1.0000    1.0000        20
               electricity|high     1.0000    0.9744    0.9870        39
            electricity|routine     0.9841    1.0000    0.9920        62
            healthcare|critical     1.0000    1.0000    1.0000        20
                healthcare|high     1.0000    1.0000    1.0000        44
             healthcare|routine     1.0000   

In [27]:
# ============================================================
# BGE-M3 ERROR ANALYSIS
# ============================================================

valid_labels = train.loc[
    valid_indices, "label"
].to_numpy()

bge_wrong = bge_pred != valid_labels

print("=== BGE-M3 ERROR ANALYSIS ===")

print(
    "Total validation errors:",
    bge_wrong.sum()
)

print(
    "Validation accuracy:",
    f"{(~bge_wrong).mean() * 100:.2f}%"
)

# ------------------------------------------------------------
# Error table
# ------------------------------------------------------------

error_df = pd.DataFrame({
    "true": valid_labels[bge_wrong],
    "predicted": bge_pred[bge_wrong]
})

print("\n=== BGE ERRORS ===")

display(
    error_df.value_counts(
        ["true", "predicted"]
    )
    .reset_index(name="count")
    .head(30)
)

# ------------------------------------------------------------
# Category / urgency error decomposition
# ------------------------------------------------------------

true_category = np.array([
    x.split("|")[0]
    for x in valid_labels
])

pred_category = np.array([
    x.split("|")[0]
    for x in bge_pred
])

true_urgency = np.array([
    x.split("|")[1]
    for x in valid_labels
])

pred_urgency = np.array([
    x.split("|")[1]
    for x in bge_pred
])

category_correct = (
    true_category == pred_category
)

urgency_correct = (
    true_urgency == pred_urgency
)

print("\n=== ERROR DECOMPOSITION ===")

print(
    "Category accuracy:",
    f"{category_correct.mean() * 100:.2f}%"
)

print(
    "Urgency accuracy:",
    f"{urgency_correct.mean() * 100:.2f}%"
)

print(
    "Both category + urgency correct:",
    f"{(category_correct & urgency_correct).mean() * 100:.2f}%"
)

print(
    "Category only correct:",
    f"{(category_correct & ~urgency_correct).sum()}"
)

print(
    "Urgency only correct:",
    f"{(~category_correct & urgency_correct).sum()}"
)

print(
    "Both wrong:",
    f"{(~category_correct & ~urgency_correct).sum()}"
)

=== BGE-M3 ERROR ANALYSIS ===
Total validation errors: 13
Validation accuracy: 98.65%

=== BGE ERRORS ===


,true,predicted,count
0,welfare_schemes|routine,welfare_schemes|high,4
1,agriculture_irrigation|high,agriculture_irrigation|routine,3
2,water_supply|high,water_supply|routine,2
3,electricity|high,electricity|routine,1
4,education|high,education|routine,1
5,law_and_order|routine,law_and_order|critical,1
6,welfare_schemes|high,welfare_schemes|routine,1



=== ERROR DECOMPOSITION ===
Category accuracy: 100.00%
Urgency accuracy: 98.65%
Both category + urgency correct: 98.65%
Category only correct: 13
Urgency only correct: 0
Both wrong: 0


In [28]:
# ============================================================
# STEP 21 — BGE-M3 EMBEDDINGS FOR ALL TRAINING DATA
# ============================================================

import numpy as np

all_train_texts = (
    train["subject"]
    .fillna("")
    .astype(str)
    + " [SEP] "
    + train["body"]
    .fillna("")
    .astype(str)
).tolist()

print("Total training texts:", len(all_train_texts))

print("\nEncoding all training data...")

all_bge_embeddings = bge_encoder.encode(
    all_train_texts,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

print("\n=== ALL-TRAIN BGE ===")

print(
    "Shape:",
    all_bge_embeddings.shape
)

print(
    "dtype:",
    all_bge_embeddings.dtype
)

print(
    "NaN:",
    np.isnan(all_bge_embeddings).any()
)

print(
    "Approx memory:",
    all_bge_embeddings.nbytes / (1024 ** 2),
    "MB"
)

Total training texts: 4800

Encoding all training data...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]


=== ALL-TRAIN BGE ===
Shape: (4800, 1024)
dtype: float32
NaN: False
Approx memory: 18.75 MB


In [29]:
# ============================================================
# STEP 22 — 5-FOLD OOF VALIDATION FOR BGE-M3
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

labels_all = train["label"].to_numpy()

N_SPLITS = 5
RANDOM_STATE = 42

oof_bge_pred = np.empty(
    len(labels_all),
    dtype=object
)

fold_results = []

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(all_bge_embeddings, labels_all),
    start=1
):

    print("\n" + "=" * 60)
    print(f"BGE-M3 OOF FOLD {fold}/{N_SPLITS}")
    print("=" * 60)

    X_tr = all_bge_embeddings[tr_idx]
    X_va = all_bge_embeddings[va_idx]

    y_tr = labels_all[tr_idx]
    y_va = labels_all[va_idx]

    clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    clf.fit(X_tr, y_tr)

    pred = clf.predict(X_va)

    acc = accuracy_score(
        y_va,
        pred
    )

    oof_bge_pred[va_idx] = pred

    fold_results.append({
        "fold": fold,
        "accuracy": acc
    })

    print(
        f"Fold accuracy: {acc * 100:.2f}%"
    )

# ------------------------------------------------------------
# Overall OOF
# ------------------------------------------------------------

results_df = pd.DataFrame(
    fold_results
)

oof_bge_accuracy = accuracy_score(
    labels_all,
    oof_bge_pred
)

print("\n" + "=" * 60)
print("BGE-M3 OOF RESULTS")
print("=" * 60)

display(
    results_df.assign(
        accuracy_pct=lambda x:
            x["accuracy"] * 100
    )
)

print(
    f"\nOverall OOF accuracy: "
    f"{oof_bge_accuracy * 100:.2f}%"
)

print(
    f"Mean fold accuracy: "
    f"{results_df['accuracy'].mean() * 100:.2f}%"
)

print(
    f"Std fold accuracy: "
    f"{results_df['accuracy'].std() * 100:.2f} pp"
)

print(
    f"\nSingle-split accuracy: "
    f"{bge_accuracy * 100:.2f}%"
)

print(
    f"OOF vs single split: "
    f"{(oof_bge_accuracy - bge_accuracy) * 100:+.2f} pp"
)


BGE-M3 OOF FOLD 1/5
Fold accuracy: 98.75%

BGE-M3 OOF FOLD 2/5
Fold accuracy: 98.23%

BGE-M3 OOF FOLD 3/5
Fold accuracy: 98.54%

BGE-M3 OOF FOLD 4/5
Fold accuracy: 98.85%

BGE-M3 OOF FOLD 5/5
Fold accuracy: 98.23%

BGE-M3 OOF RESULTS


,fold,accuracy,accuracy_pct
0,1,0.987500,98.750000
1,2,0.982292,98.229167
2,3,0.985417,98.541667
3,4,0.988542,98.854167
4,5,0.982292,98.229167



Overall OOF accuracy: 98.52%
Mean fold accuracy: 98.52%
Std fold accuracy: 0.29 pp

Single-split accuracy: 98.65%
OOF vs single split: -0.12 pp


In [30]:
[name for name in globals() if "oof" in name.lower()]

['oof_pred', 'oof_accuracy', 'oof_bge_pred', 'oof_bge_accuracy']

In [32]:
print(f"Agreement: {agreement.mean() * 100:.2f}%")
print(f"Disagreement: {(~agreement).mean() * 100:.2f}%")

# ------------------------------------------------------------
# Disagreement outcomes
# ------------------------------------------------------------

disagree = ~agreement

print("\n=== WHEN THEY DISAGREE ===")

print(
    "Total disagreements:",
    disagree.sum()
)

print(
    "TF-IDF correct:",
    (tfidf_correct & disagree).sum()
)

print(
    "BGE-M3 correct:",
    (bge_correct & disagree).sum()
)

print(
    "Both wrong:",
    (~tfidf_correct & ~bge_correct & disagree).sum()
)

# ------------------------------------------------------------
# BGE errors that TF-IDF can rescue
# ------------------------------------------------------------

bge_wrong = ~bge_correct

tfidf_rescues_bge = (
    bge_wrong
    & tfidf_correct
)

bge_only_correct = (
    bge_correct
    & ~tfidf_correct
)

both_wrong = (
    ~bge_correct
    & ~tfidf_correct
)

print("\n=== BGE ERROR ANALYSIS ===")

print(
    "BGE errors:",
    bge_wrong.sum()
)

print(
    "TF-IDF rescues BGE:",
    tfidf_rescues_bge.sum()
)

print(
    "BGE correct / TF-IDF wrong:",
    bge_only_correct.sum()
)

print(
    "Both wrong:",
    both_wrong.sum()
)

# ------------------------------------------------------------
# Oracle ceiling
# ------------------------------------------------------------

oracle_correct = (
    bge_correct | tfidf_correct
)

oracle_accuracy = oracle_correct.mean()

print("\n=== ORACLE CEILING ===")

print(
    "At least one model correct:",
    oracle_correct.sum()
)

print(
    f"Oracle accuracy: "
    f"{oracle_accuracy * 100:.2f}%"
)

print(
    f"Potential headroom over BGE: "
    f"{(oracle_accuracy - bge_acc) * 100:+.2f} pp"
)

# ------------------------------------------------------------
# Detailed disagreement table
# ------------------------------------------------------------

disagreement_df = pd.DataFrame({
    "true": true_oof[disagree],
    "tfidf": oof_pred[disagree],
    "bge": oof_bge_pred[disagree],
    "tfidf_correct": tfidf_correct[disagree],
    "bge_correct": bge_correct[disagree]
})

print("\n=== DISAGREEMENT PATTERNS ===")

display(
    disagreement_df
    .groupby(
        ["tfidf", "bge", "true"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

Agreement: 94.75%
Disagreement: 5.25%

=== WHEN THEY DISAGREE ===
Total disagreements: 252
TF-IDF correct: 22
BGE-M3 correct: 230
Both wrong: 0

=== BGE ERROR ANALYSIS ===
BGE errors: 71
TF-IDF rescues BGE: 22
BGE correct / TF-IDF wrong: 230
Both wrong: 49

=== ORACLE CEILING ===
At least one model correct: 4751
Oracle accuracy: 98.98%
Potential headroom over BGE: +0.46 pp

=== DISAGREEMENT PATTERNS ===


,tfidf,bge,true,count
52,welfare_schemes|high,welfare_schemes|routine,welfare_schemes|routine,15
38,roads_transport|routine,roads_transport|high,roads_transport|high,11
15,electricity|routine,electricity|high,electricity|high,10
6,education|high,education|routine,education|routine,9
34,roads_transport|high,roads_transport|critical,roads_transport|critical,9
36,roads_transport|routine,roads_transport|critical,roads_transport|critical,9
8,education|routine,education|high,education|high,9
46,water_supply|routine,water_supply|high,water_supply|high,8
45,water_supply|routine,water_supply|critical,water_supply|critical,8
30,law_and_order|routine,law_and_order|high,law_and_order|high,8


In [33]:
# ============================================================
# STEP 24 — BGE OOF CONFIDENCE / ERROR ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

# Recreate OOF BGE decision scores
# using the same 5-fold structure.
#
# We need scores for every OOF row, so retrain the cheap
# LinearSVC models fold-by-fold.

labels_all = train["label"].to_numpy()

oof_bge_scores = np.zeros(
    (len(labels_all), len(np.unique(labels_all))),
    dtype=np.float32
)

# Classes must be consistent across folds
global_classes = np.sort(
    np.unique(labels_all)
)

class_to_col = {
    c: i
    for i, c in enumerate(global_classes)
}

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(all_bge_embeddings, labels_all),
    start=1
):

    clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    clf.fit(
        all_bge_embeddings[tr_idx],
        labels_all[tr_idx]
    )

    fold_scores = clf.decision_function(
        all_bge_embeddings[va_idx]
    )

    # clf.classes_ should contain all 24 classes
    for local_col, cls in enumerate(clf.classes_):
        global_col = class_to_col[cls]
        oof_bge_scores[
            va_idx,
            global_col
        ] = fold_scores[:, local_col]

# ------------------------------------------------------------
# Calculate winning margin
# ------------------------------------------------------------

sorted_scores = np.sort(
    oof_bge_scores,
    axis=1
)

bge_margin = (
    sorted_scores[:, -1]
    - sorted_scores[:, -2]
)

bge_correct = (
    oof_bge_pred == labels_all
)

print("=== BGE OOF CONFIDENCE ===")

print(
    f"Minimum margin: "
    f"{bge_margin.min():.4f}"
)

print(
    f"Median margin: "
    f"{np.median(bge_margin):.4f}"
)

print(
    f"Mean margin: "
    f"{bge_margin.mean():.4f}"
)

print(
    f"Maximum margin: "
    f"{bge_margin.max():.4f}"
)

print("\n=== CONFIDENCE BY CORRECTNESS ===")

print(
    "Correct median margin:",
    f"{np.median(bge_margin[bge_correct]):.4f}"
)

print(
    "Wrong median margin:",
    f"{np.median(bge_margin[~bge_correct]):.4f}"
)

# ------------------------------------------------------------
# Error rate by confidence bucket
# ------------------------------------------------------------

confidence_df = pd.DataFrame({
    "margin": bge_margin,
    "correct": bge_correct
})

confidence_df["bucket"] = pd.qcut(
    confidence_df["margin"],
    q=10,
    duplicates="drop"
)

bucket_results = (
    confidence_df
    .groupby("bucket", observed=True)
    .agg(
        samples=("correct", "size"),
        accuracy=("correct", "mean"),
        median_margin=("margin", "median")
    )
    .reset_index()
)

bucket_results["accuracy_pct"] = (
    bucket_results["accuracy"] * 100
)

print("\n=== ACCURACY BY BGE CONFIDENCE ===")

display(
    bucket_results[
        [
            "bucket",
            "samples",
            "accuracy_pct",
            "median_margin"
        ]
    ]
)

=== BGE OOF CONFIDENCE ===
Minimum margin: 0.0001
Median margin: 1.8225
Mean margin: 1.7022
Maximum margin: 3.4488

=== CONFIDENCE BY CORRECTNESS ===
Correct median margin: 1.8282
Wrong median margin: 0.1535

=== ACCURACY BY BGE CONFIDENCE ===


,bucket,samples,accuracy_pct,median_margin
0,"(-0.0009252, 0.75]",480,85.208333,0.409534
1,"(0.75, 1.381]",480,100.000000,1.115340
2,"(1.381, 1.593]",480,100.000000,1.504977
3,"(1.593, 1.723]",480,100.000000,1.665774
4,"(1.723, 1.823]",480,100.000000,1.774642
5,"(1.823, 1.909]",480,100.000000,1.867652
6,"(1.909, 2.005]",480,100.000000,1.953462
7,"(2.005, 2.119]",480,100.000000,2.055892
8,"(2.119, 2.293]",480,100.000000,2.192671
9,"(2.293, 3.449]",480,100.000000,2.442708


In [34]:
# ============================================================
# STEP 25 — CONFIDENCE-GATED TF-IDF RESCUE
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Recreate TF-IDF OOF decision scores
# ------------------------------------------------------------

labels_all = train["label"].to_numpy()

global_classes = np.sort(
    np.unique(labels_all)
)

class_to_col = {
    c: i
    for i, c in enumerate(global_classes)
}

oof_tfidf_scores = np.zeros(
    (len(labels_all), len(global_classes)),
    dtype=np.float32
)

# IMPORTANT:
# This assumes the same TF-IDF configuration used to create
# oof_pred earlier:
#
# word + char TF-IDF
# LinearSVC(C=1.0)
#
# We refit within each fold so the scores are genuinely OOF.

texts_all = (
    train["body"]
    .fillna("")
    .astype(str)
    .to_numpy()
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(texts_all, labels_all),
    start=1
):

    print(f"Generating TF-IDF scores: fold {fold}/5")

    tr_text = texts_all[tr_idx]
    va_text = texts_all[va_idx]

    # Word TF-IDF
    word_vec = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True,
        max_features=120_000
    )

    X_word_tr = word_vec.fit_transform(
        tr_text
    )

    X_word_va = word_vec.transform(
        va_text
    )

    # Character TF-IDF
    char_vec = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        sublinear_tf=True,
        max_features=120_000
    )

    X_char_tr = char_vec.fit_transform(
        tr_text
    )

    X_char_va = char_vec.transform(
        va_text
    )

    X_tr = hstack([
        X_word_tr,
        X_char_tr
    ])

    X_va = hstack([
        X_word_va,
        X_char_va
    ])

    clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    clf.fit(
        X_tr,
        labels_all[tr_idx]
    )

    scores = clf.decision_function(
        X_va
    )

    for local_col, cls in enumerate(
        clf.classes_
    ):
        global_col = class_to_col[cls]

        oof_tfidf_scores[
            va_idx,
            global_col
        ] = scores[:, local_col]

# ------------------------------------------------------------
# 2. Calculate TF-IDF margins
# ------------------------------------------------------------

sorted_tfidf = np.sort(
    oof_tfidf_scores,
    axis=1
)

tfidf_margin = (
    sorted_tfidf[:, -1]
    - sorted_tfidf[:, -2]
)

print("\nTF-IDF confidence calculated.")

print(
    "Median TF-IDF margin:",
    f"{np.median(tfidf_margin):.4f}"
)

# ------------------------------------------------------------
# 3. BGE margin already exists
# ------------------------------------------------------------

print(
    "Median BGE margin:",
    f"{np.median(bge_margin):.4f}"
)

# ------------------------------------------------------------
# 4. Test confidence thresholds
# ------------------------------------------------------------

thresholds = [
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.50,
    0.60,
    0.75,
    1.00
]

results = []

bge_base = oof_bge_pred.copy()

for threshold in thresholds:

    hybrid = bge_base.copy()

    uncertain = (
        bge_margin < threshold
    )

    # For uncertain BGE predictions,
    # replace with TF-IDF prediction.
    tfidf_pred_from_scores = (
        global_classes[
            np.argmax(
                oof_tfidf_scores,
                axis=1
            )
        ]
    )

    hybrid[uncertain] = (
        tfidf_pred_from_scores[uncertain]
    )

    acc = accuracy_score(
        labels_all,
        hybrid
    )

    changed = (
        hybrid != bge_base
    )

    results.append({
        "threshold": threshold,
        "accuracy": acc,
        "accuracy_pct": acc * 100,
        "rows_consulted": int(
            uncertain.sum()
        ),
        "predictions_changed": int(
            changed.sum()
        )
    })

results_df = pd.DataFrame(
    results
)

print("\n=== CONFIDENCE-GATED RESULTS ===")

display(
    results_df[
        [
            "threshold",
            "accuracy_pct",
            "rows_consulted",
            "predictions_changed"
        ]
    ]
)

# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

best_idx = results_df[
    "accuracy"
].idxmax()

best = results_df.loc[
    best_idx
]

print("\n=== BEST GATED RESULT ===")

print(
    f"Threshold: {best['threshold']}"
)

print(
    f"Accuracy: {best['accuracy_pct']:.2f}%"
)

print(
    f"Rows consulted: "
    f"{best['rows_consulted']}"
)

print(
    f"Predictions changed: "
    f"{best['predictions_changed']}"
)

print(
    f"Improvement over BGE: "
    f"{best['accuracy_pct'] - oof_bge_accuracy * 100:+.2f} pp"
)

Generating TF-IDF scores: fold 1/5
Generating TF-IDF scores: fold 2/5
Generating TF-IDF scores: fold 3/5
Generating TF-IDF scores: fold 4/5
Generating TF-IDF scores: fold 5/5

TF-IDF confidence calculated.
Median TF-IDF margin: 2.0836
Median BGE margin: 1.8225

=== CONFIDENCE-GATED RESULTS ===


,threshold,accuracy_pct,rows_consulted,predictions_changed
0,0.05,98.437500,25,14
1,0.10,98.354167,54,30
2,0.15,98.333333,77,37
3,0.20,98.166667,108,49
4,0.25,98.062500,127,54
5,0.30,97.791667,157,71
6,0.35,97.395833,191,90
7,0.40,97.145833,232,102
8,0.50,96.375000,312,139
9,0.60,95.937500,382,162



=== BEST GATED RESULT ===
Threshold: 0.05
Accuracy: 98.44%
Rows consulted: 25.0
Predictions changed: 14.0
Improvement over BGE: -0.08 pp


In [35]:
# ============================================================
# STEP 26 — BGE OOF ERROR ANALYSIS
# ============================================================

error_mask = (
    oof_bge_pred != labels_all
)

errors = pd.DataFrame({
    "true": labels_all[error_mask],
    "predicted": oof_bge_pred[error_mask]
})

errors["true_category"] = (
    errors["true"]
    .str.split("|")
    .str[0]
)

errors["true_urgency"] = (
    errors["true"]
    .str.split("|")
    .str[1]
)

errors["pred_category"] = (
    errors["predicted"]
    .str.split("|")
    .str[0]
)

errors["pred_urgency"] = (
    errors["predicted"]
    .str.split("|")
    .str[1]
)

print("=== BGE OOF ERRORS ===")

print(
    "Total errors:",
    len(errors)
)

print("\n=== ERROR TYPE ===")

errors["error_type"] = np.where(
    errors["true_category"]
    == errors["pred_category"],
    "urgency_only",
    "category_error"
)

display(
    errors["error_type"]
    .value_counts()
)

print("\n=== TRUE → PREDICTED URGENCY ===")

display(
    errors
    .groupby(
        [
            "true_urgency",
            "pred_urgency"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

print("\n=== CATEGORY-SPECIFIC URGENCY ERRORS ===")

display(
    errors[
        errors["error_type"]
        == "urgency_only"
    ]
    .groupby(
        [
            "true_category",
            "true_urgency",
            "pred_urgency"
        ]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
)

=== BGE OOF ERRORS ===
Total errors: 71

=== ERROR TYPE ===


error_type
urgency_only    71
Name: count, dtype: int64


=== TRUE → PREDICTED URGENCY ===


,true_urgency,pred_urgency,count
3,high,routine,21
5,routine,high,19
1,critical,routine,9
0,critical,high,8
2,high,critical,7
4,routine,critical,7



=== CATEGORY-SPECIFIC URGENCY ERRORS ===


,true_category,true_urgency,pred_urgency,count
29,welfare_schemes,routine,high,8
27,welfare_schemes,high,routine,7
1,agriculture_irrigation,high,routine,4
23,roads_transport,routine,high,4
8,electricity,high,routine,4
10,healthcare,critical,routine,3
15,law_and_order,critical,high,3
2,education,high,critical,3
20,roads_transport,critical,high,3
9,electricity,routine,high,2


In [36]:
# ============================================================
# STEP 27 — INSPECT WELFARE URGENCY ERRORS
# ============================================================

welfare_errors = errors[
    (errors["true_category"] == "welfare_schemes")
]

print("Welfare urgency errors:")
print(len(welfare_errors))

# Get original row indices
welfare_error_indices = np.where(
    error_mask
    & np.array([
        label.split("|")[0] == "welfare_schemes"
        for label in labels_all
    ])
)[0]

inspection = train.iloc[
    welfare_error_indices
][
    [
        "subject",
        "body",
        "channel",
        "district",
        "complaint_history",
        "label"
    ]
].copy()

inspection["bge_prediction"] = (
    oof_bge_pred[welfare_error_indices]
)

display(
    inspection.to_string(index=False)
)

Welfare urgency errors:
17


'                                            subject                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           body        channel    district complaint_history                   label           bge_prediction\n    Ration card application pending isnce lats week                                                                                                                               The second installment of the housing scheme has not been released though the walls are complete and photos were uploaded. The half-built house is 

In [37]:
# ============================================================
# STEP 28 — BGE-M3 + METADATA
# ============================================================

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

labels_all = train["label"].to_numpy()

metadata_cols = [
    "channel",
    "district",
    "complaint_history"
]

metadata = (
    train[metadata_cols]
    .fillna("MISSING")
    .astype(str)
)

print("Metadata columns:")
print(metadata_cols)

print("\nUnique values:")
for col in metadata_cols:
    print(
        f"{col}:",
        metadata[col].nunique()
    )

# ------------------------------------------------------------
# One-hot encode metadata
# ------------------------------------------------------------

encoder_meta = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_meta = encoder_meta.fit_transform(
    metadata
)

print("\nMetadata matrix:")
print(X_meta.shape)

# ------------------------------------------------------------
# Combine BGE + metadata
# ------------------------------------------------------------

X_bge_meta = hstack([
    csr_matrix(all_bge_embeddings),
    X_meta
]).tocsr()

print("\nCombined matrix:")
print(X_bge_meta.shape)

# ------------------------------------------------------------
# 5-fold OOF
# ------------------------------------------------------------

oof_bge_meta_pred = np.empty(
    len(labels_all),
    dtype=object
)

fold_results = []

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(X_bge_meta, labels_all),
    start=1
):

    print(
        f"\nTraining BGE + metadata fold "
        f"{fold}/5..."
    )

    clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    clf.fit(
        X_bge_meta[tr_idx],
        labels_all[tr_idx]
    )

    pred = clf.predict(
        X_bge_meta[va_idx]
    )

    acc = accuracy_score(
        labels_all[va_idx],
        pred
    )

    oof_bge_meta_pred[va_idx] = pred

    fold_results.append(acc)

    print(
        f"Fold accuracy: {acc * 100:.2f}%"
    )

# ------------------------------------------------------------
# Overall result
# ------------------------------------------------------------

bge_meta_oof_accuracy = accuracy_score(
    labels_all,
    oof_bge_meta_pred
)

print("\n" + "=" * 60)
print("BGE + METADATA OOF RESULT")
print("=" * 60)

print(
    f"Overall OOF accuracy: "
    f"{bge_meta_oof_accuracy * 100:.2f}%"
)

print(
    f"Mean fold accuracy: "
    f"{np.mean(fold_results) * 100:.2f}%"
)

print(
    f"Std fold accuracy: "
    f"{np.std(fold_results, ddof=1) * 100:.2f} pp"
)

print(
    f"\nBGE-M3 alone: "
    f"{oof_bge_accuracy * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(bge_meta_oof_accuracy - oof_bge_accuracy) * 100:+.2f} pp"
)

Metadata columns:
['channel', 'district', 'complaint_history']

Unique values:
channel: 5
district: 8
complaint_history: 3

Metadata matrix:
(4800, 16)

Combined matrix:
(4800, 1040)

Training BGE + metadata fold 1/5...
Fold accuracy: 98.85%

Training BGE + metadata fold 2/5...
Fold accuracy: 98.85%

Training BGE + metadata fold 3/5...
Fold accuracy: 98.54%

Training BGE + metadata fold 4/5...
Fold accuracy: 98.75%

Training BGE + metadata fold 5/5...
Fold accuracy: 98.12%

BGE + METADATA OOF RESULT
Overall OOF accuracy: 98.62%
Mean fold accuracy: 98.62%
Std fold accuracy: 0.31 pp

BGE-M3 alone: 98.52%
Improvement: +0.10 pp


In [38]:
# ============================================================
# STEP 29 — BGE vs BGE+METADATA OOF
# ============================================================

true_all = labels_all

bge_correct = (
    oof_bge_pred == true_all
)

meta_correct = (
    oof_bge_meta_pred == true_all
)

changed = (
    oof_bge_meta_pred != oof_bge_pred
)

print("=== BGE vs BGE + METADATA ===")

print(
    f"BGE OOF:          "
    f"{bge_correct.mean() * 100:.2f}%"
)

print(
    f"BGE + metadata:   "
    f"{meta_correct.mean() * 100:.2f}%"
)

print(
    f"Changed:          "
    f"{changed.sum()}"
)

print(
    f"BGE correct:      "
    f"{(bge_correct & changed).sum()}"
)

print(
    f"Metadata correct: "
    f"{(meta_correct & changed).sum()}"
)

print(
    f"Gained:           "
    f"{(~bge_correct & meta_correct & changed).sum()}"
)

print(
    f"Lost:             "
    f"{(bge_correct & ~meta_correct & changed).sum()}"
)

print(
    f"Both wrong:       "
    f"{(~bge_correct & ~meta_correct & changed).sum()}"
)

# ------------------------------------------------------------
# Changed prediction patterns
# ------------------------------------------------------------

changed_df = pd.DataFrame({
    "true": true_all[changed],
    "bge": oof_bge_pred[changed],
    "bge_meta": oof_bge_meta_pred[changed]
})

print("\n=== CHANGED PREDICTION PATTERNS ===")

display(
    changed_df
    .groupby(
        ["true", "bge", "bge_meta"]
    )
    .size()
    .reset_index(name="count")
    .sort_values(
        "count",
        ascending=False
    )
    .head(30)
)

=== BGE vs BGE + METADATA ===
BGE OOF:          98.52%
BGE + metadata:   98.62%
Changed:          41
BGE correct:      18
Metadata correct: 23
Gained:           23
Lost:             18
Both wrong:       0

=== CHANGED PREDICTION PATTERNS ===


,true,bge,bge_meta,count
31,welfare_schemes|high,welfare_schemes|routine,welfare_schemes|high,4
13,healthcare|routine,healthcare|routine,healthcare|high,2
24,roads_transport|routine,roads_transport|high,roads_transport|routine,2
11,healthcare|routine,healthcare|critical,healthcare|routine,2
21,roads_transport|critical,roads_transport|critical,roads_transport|routine,2
0,agriculture_irrigation|critical,agriculture_irrigation|routine,agriculture_irrigation|critical,1
1,agriculture_irrigation|high,agriculture_irrigation|high,agriculture_irrigation|routine,1
6,electricity|high,electricity|routine,electricity|high,1
2,education|high,education|critical,education|high,1
3,education|high,education|high,education|routine,1


In [39]:
# ============================================================
# STEP 30 — BGE + METADATA ERROR DECOMPOSITION
# ============================================================

true_cat = np.array([
    x.split("|")[0]
    for x in labels_all
])

pred_cat = np.array([
    x.split("|")[0]
    for x in oof_bge_meta_pred
])

true_urg = np.array([
    x.split("|")[1]
    for x in labels_all
])

pred_urg = np.array([
    x.split("|")[1]
    for x in oof_bge_meta_pred
])

cat_correct = true_cat == pred_cat
urg_correct = true_urg == pred_urg

print("=== BGE + METADATA DECOMPOSITION ===")

print(
    f"Category accuracy: "
    f"{cat_correct.mean() * 100:.2f}%"
)

print(
    f"Urgency accuracy:  "
    f"{urg_correct.mean() * 100:.2f}%"
)

print(
    f"Full-label accuracy:"
    f" {(cat_correct & urg_correct).mean() * 100:.2f}%"
)

print("\n=== ERROR TYPES ===")

print(
    "Category only correct:",
    (cat_correct & ~urg_correct).sum()
)

print(
    "Urgency only correct:",
    (~cat_correct & urg_correct).sum()
)

print(
    "Both wrong:",
    (~cat_correct & ~urg_correct).sum()
)

print("\n=== URGENCY CONFUSION ===")

urgency_conf = pd.crosstab(
    pd.Series(true_urg, name="true"),
    pd.Series(pred_urg, name="pred")
)

display(urgency_conf)

=== BGE + METADATA DECOMPOSITION ===
Category accuracy: 100.00%
Urgency accuracy:  98.62%
Full-label accuracy: 98.62%

=== ERROR TYPES ===
Category only correct: 66
Urgency only correct: 0
Both wrong: 0

=== URGENCY CONFUSION ===


pred,critical,high,routine
true,,,
critical,851,10,8
high,6,1734,16
routine,7,19,2149


In [40]:
# ============================================================
# STEP 31 — DEDICATED BGE + METADATA URGENCY OOF
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

labels_all = train["label"].to_numpy()

# ------------------------------------------------------------
# True category / urgency
# ------------------------------------------------------------

true_categories = np.array([
    x.split("|")[0]
    for x in labels_all
])

true_urgencies = np.array([
    x.split("|")[1]
    for x in labels_all
])

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

oof_dedicated_urgency = np.empty(
    len(labels_all),
    dtype=object
)

oof_dedicated_category = np.empty(
    len(labels_all),
    dtype=object
)

fold_results = []

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# ------------------------------------------------------------
# OOF
# ------------------------------------------------------------

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(X_bge_meta, labels_all),
    start=1
):

    print("\n" + "=" * 60)
    print(f"DEDICATED URGENCY FOLD {fold}/5")
    print("=" * 60)

    # --------------------------------------------------------
    # Dedicated urgency classifier
    # --------------------------------------------------------

    urgency_clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    urgency_clf.fit(
        X_bge_meta[tr_idx],
        true_urgencies[tr_idx]
    )

    urgency_pred = urgency_clf.predict(
        X_bge_meta[va_idx]
    )

    # --------------------------------------------------------
    # Keep category from the BGE + metadata 24-class model
    #
    # Train it only on the training portion of this fold.
    # --------------------------------------------------------

    category_clf = LinearSVC(
        C=1.0,
        class_weight=None
    )

    category_clf.fit(
        X_bge_meta[tr_idx],
        labels_all[tr_idx]
    )

    label_pred = category_clf.predict(
        X_bge_meta[va_idx]
    )

    category_pred = np.array([
        x.split("|")[0]
        for x in label_pred
    ])

    # --------------------------------------------------------
    # Combine category + dedicated urgency
    # --------------------------------------------------------

    hybrid_pred = np.array([
        f"{cat}|{urg}"
        for cat, urg in zip(
            category_pred,
            urgency_pred
        )
    ])

    oof_dedicated_urgency[va_idx] = (
        urgency_pred
    )

    oof_dedicated_category[va_idx] = (
        category_pred
    )

    fold_acc = accuracy_score(
        labels_all[va_idx],
        hybrid_pred
    )

    urgency_acc = accuracy_score(
        true_urgencies[va_idx],
        urgency_pred
    )

    category_acc = accuracy_score(
        true_categories[va_idx],
        category_pred
    )

    fold_results.append({
        "fold": fold,
        "full_accuracy": fold_acc,
        "urgency_accuracy": urgency_acc,
        "category_accuracy": category_acc
    })

    print(
        f"Full-label accuracy: "
        f"{fold_acc * 100:.2f}%"
    )

    print(
        f"Urgency accuracy: "
        f"{urgency_acc * 100:.2f}%"
    )

    print(
        f"Category accuracy: "
        f"{category_acc * 100:.2f}%"
    )

# ------------------------------------------------------------
# Overall OOF predictions
# ------------------------------------------------------------

final_dedicated_oof = np.array([
    f"{cat}|{urg}"
    for cat, urg in zip(
        oof_dedicated_category,
        oof_dedicated_urgency
    )
])

dedicated_oof_accuracy = accuracy_score(
    labels_all,
    final_dedicated_oof
)

dedicated_urgency_accuracy = accuracy_score(
    true_urgencies,
    oof_dedicated_urgency
)

dedicated_category_accuracy = accuracy_score(
    true_categories,
    oof_dedicated_category
)

results_df = pd.DataFrame(
    fold_results
)

print("\n" + "=" * 60)
print("DEDICATED BGE URGENCY — OOF RESULT")
print("=" * 60)

display(
    results_df.assign(
        full_accuracy_pct=lambda x:
            x["full_accuracy"] * 100,
        urgency_accuracy_pct=lambda x:
            x["urgency_accuracy"] * 100,
        category_accuracy_pct=lambda x:
            x["category_accuracy"] * 100
    )
)

print(
    f"\nFull-label OOF accuracy: "
    f"{dedicated_oof_accuracy * 100:.2f}%"
)

print(
    f"Urgency OOF accuracy: "
    f"{dedicated_urgency_accuracy * 100:.2f}%"
)

print(
    f"Category OOF accuracy: "
    f"{dedicated_category_accuracy * 100:.2f}%"
)

print(
    f"\nCurrent BGE + metadata: "
    f"{bge_meta_oof_accuracy * 100:.2f}%"
)

print(
    f"Improvement: "
    f"{(dedicated_oof_accuracy - bge_meta_oof_accuracy) * 100:+.2f} pp"
)


DEDICATED URGENCY FOLD 1/5
Full-label accuracy: 98.44%
Urgency accuracy: 98.44%
Category accuracy: 100.00%

DEDICATED URGENCY FOLD 2/5
Full-label accuracy: 98.12%
Urgency accuracy: 98.12%
Category accuracy: 100.00%

DEDICATED URGENCY FOLD 3/5
Full-label accuracy: 98.33%
Urgency accuracy: 98.33%
Category accuracy: 100.00%

DEDICATED URGENCY FOLD 4/5
Full-label accuracy: 98.33%
Urgency accuracy: 98.33%
Category accuracy: 100.00%

DEDICATED URGENCY FOLD 5/5
Full-label accuracy: 98.12%
Urgency accuracy: 98.12%
Category accuracy: 100.00%

DEDICATED BGE URGENCY — OOF RESULT


,fold,full_accuracy,urgency_accuracy,category_accuracy,full_accuracy_pct,urgency_accuracy_pct,category_accuracy_pct
0,1,0.984375,0.984375,1.0,98.437500,98.437500,100.0
1,2,0.981250,0.981250,1.0,98.125000,98.125000,100.0
2,3,0.983333,0.983333,1.0,98.333333,98.333333,100.0
3,4,0.983333,0.983333,1.0,98.333333,98.333333,100.0
4,5,0.981250,0.981250,1.0,98.125000,98.125000,100.0



Full-label OOF accuracy: 98.27%
Urgency OOF accuracy: 98.27%
Category OOF accuracy: 100.00%

Current BGE + metadata: 98.62%
Improvement: -0.35 pp


In [41]:
# ============================================================
# FINAL MODEL — BGE-M3 + METADATA
# Train on all 4,800 rows
# Predict all 2,000 test rows
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import LinearSVC

print("=" * 70)
print("FINAL BGE-M3 + METADATA MODEL")
print("=" * 70)

# ------------------------------------------------------------
# 1. BUILD TRAIN / TEST TEXT
# ------------------------------------------------------------

train_texts_final = (
    train["subject"]
    .fillna("")
    .astype(str)
    + " [SEP] "
    + train["body"]
    .fillna("")
    .astype(str)
).tolist()

test_texts_final = (
    test["subject"]
    .fillna("")
    .astype(str)
    + " [SEP] "
    + test["body"]
    .fillna("")
    .astype(str)
).tolist()

print("\nTrain texts:", len(train_texts_final))
print("Test texts:", len(test_texts_final))

# ------------------------------------------------------------
# 2. ENCODE ALL TRAIN + TEST WITH BGE-M3
# ------------------------------------------------------------

print("\nEncoding full training data...")

final_train_bge = bge_encoder.encode(
    train_texts_final,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

print("\nEncoding test data...")

final_test_bge = bge_encoder.encode(
    test_texts_final,
    batch_size=32,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
).astype(np.float32)

print("\nBGE shapes:")
print("Train:", final_train_bge.shape)
print("Test :", final_test_bge.shape)

# ------------------------------------------------------------
# 3. METADATA
# ------------------------------------------------------------

metadata_cols = [
    "channel",
    "district",
    "complaint_history"
]

train_metadata = (
    train[metadata_cols]
    .fillna("MISSING")
    .astype(str)
)

test_metadata = (
    test[metadata_cols]
    .fillna("MISSING")
    .astype(str)
)

# IMPORTANT:
# Fit encoder ONLY on training data.
# Test gets transform() so unseen categories are handled safely.

final_meta_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

X_train_meta = final_meta_encoder.fit_transform(
    train_metadata
)

X_test_meta = final_meta_encoder.transform(
    test_metadata
)

print("\nMetadata shapes:")
print("Train:", X_train_meta.shape)
print("Test :", X_test_meta.shape)

# ------------------------------------------------------------
# 4. COMBINE BGE + METADATA
# ------------------------------------------------------------

X_final_train = hstack([
    csr_matrix(final_train_bge),
    X_train_meta
]).tocsr()

X_final_test = hstack([
    csr_matrix(final_test_bge),
    X_test_meta
]).tocsr()

print("\nFinal feature matrices:")
print("Train:", X_final_train.shape)
print("Test :", X_final_test.shape)

# ------------------------------------------------------------
# 5. TRAIN FINAL 24-CLASS CLASSIFIER
# ------------------------------------------------------------

print("\nTraining final LinearSVC...")

final_clf = LinearSVC(
    C=1.0,
    class_weight=None
)

final_clf.fit(
    X_final_train,
    train["label"].to_numpy()
)

print("Final classifier trained.")

# ------------------------------------------------------------
# 6. TEST PREDICTIONS
# ------------------------------------------------------------

print("\nGenerating test predictions...")

final_test_predictions = final_clf.predict(
    X_final_test
)

print(
    "Number of predictions:",
    len(final_test_predictions)
)

print("\nFirst 20 predictions:")

for i, pred in enumerate(
    final_test_predictions[:20]
):
    print(
        f"{i:4d}  {test.iloc[i]['id']}  {pred}"
    )

# ------------------------------------------------------------
# 7. VALIDATE PREDICTIONS
# ------------------------------------------------------------

valid_labels = set(
    train["label"].unique()
)

print("\n" + "=" * 70)
print("SUBMISSION VALIDATION")
print("=" * 70)

print(
    "Expected rows:",
    len(test)
)

print(
    "Predicted rows:",
    len(final_test_predictions)
)

print(
    "Missing predictions:",
    pd.isna(final_test_predictions).sum()
)

print(
    "Unique predicted labels:",
    len(np.unique(final_test_predictions))
)

invalid_predictions = [
    x for x in final_test_predictions
    if x not in valid_labels
]

print(
    "Invalid labels:",
    len(invalid_predictions)
)

print(
    "Unique test IDs:",
    test["id"].nunique()
)

print(
    "Duplicate test IDs:",
    test["id"].duplicated().sum()
)

# ------------------------------------------------------------
# 8. CREATE SUBMISSION
# ------------------------------------------------------------

submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "label": final_test_predictions
})

submission_path = "/kaggle/working/submission_bge_metadata.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("\n" + "=" * 70)
print("SUBMISSION CREATED")
print("=" * 70)

print(
    "Path:",
    submission_path
)

print(
    "Shape:",
    submission.shape
)

print("\nSubmission head:")

display(
    submission.head(10)
)

print("\nLabel distribution:")

display(
    submission["label"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 9. FINAL ASSERTIONS
# ------------------------------------------------------------

assert len(submission) == 2000
assert submission["id"].notna().all()
assert submission["label"].notna().all()
assert submission["id"].duplicated().sum() == 0
assert set(submission["label"]).issubset(valid_labels)

print("\n✅ ALL SUBMISSION CHECKS PASSED")
print("✅ 2,000 predictions")
print("✅ No missing IDs")
print("✅ No missing labels")
print("✅ No duplicate IDs")
print("✅ All labels valid")
print("\nReady for Kaggle submission.")

FINAL BGE-M3 + METADATA MODEL

Train texts: 4800
Test texts: 2000

Encoding full training data...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]


Encoding test data...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]


BGE shapes:
Train: (4800, 1024)
Test : (2000, 1024)

Metadata shapes:
Train: (4800, 16)
Test : (2000, 16)

Final feature matrices:
Train: (4800, 1040)
Test : (2000, 1040)

Training final LinearSVC...
Final classifier trained.

Generating test predictions...
Number of predictions: 2000

First 20 predictions:
   0  G105529  agriculture_irrigation|high
   1  G105472  electricity|routine
   2  G103379  roads_transport|high
   3  G101632  water_supply|critical
   4  G106613  roads_transport|critical
   5  G105736  electricity|high
   6  G103228  roads_transport|routine
   7  G103804  law_and_order|routine
   8  G106428  welfare_schemes|routine
   9  G104133  roads_transport|routine
  10  G106744  roads_transport|routine
  11  G100372  agriculture_irrigation|critical
  12  G100616  electricity|routine
  13  G103407  agriculture_irrigation|routine
  14  G105567  education|routine
  15  G100550  law_and_order|high
  16  G101283  welfare_schemes|routine
  17  G100896  roads_transport|routine
 

,id,label
0,G105529,agriculture_irrigation|high
1,G105472,electricity|routine
2,G103379,roads_transport|high
3,G101632,water_supply|critical
4,G106613,roads_transport|critical
5,G105736,electricity|high
6,G103228,roads_transport|routine
7,G103804,law_and_order|routine
8,G106428,welfare_schemes|routine
9,G104133,roads_transport|routine



Label distribution:


label
agriculture_irrigation|critical     38
agriculture_irrigation|high         75
agriculture_irrigation|routine     109
education|critical                  38
education|high                      70
education|routine                   88
electricity|critical                42
electricity|high                    84
electricity|routine                126
healthcare|critical                 39
healthcare|high                     96
healthcare|routine                 108
law_and_order|critical              38
law_and_order|high                  81
law_and_order|routine              101
roads_transport|critical            57
roads_transport|high               120
roads_transport|routine            145
water_supply|critical               51
water_supply|high                  101
water_supply|routine               123
welfare_schemes|critical            50
welfare_schemes|high               113
welfare_schemes|routine            107
Name: count, dtype: int64


✅ ALL SUBMISSION CHECKS PASSED
✅ 2,000 predictions
✅ No missing IDs
✅ No missing labels
✅ No duplicate IDs
✅ All labels valid

Ready for Kaggle submission.
